In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

for path in sorted(Path("/kaggle/input").rglob("*.json")):
    print(path)

In [ ]:
import json
from pathlib import Path

annotation_files = list(
    Path("/kaggle/input").rglob(
        "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    )
)

assert len(annotation_files) == 1, "Expected one annotation file"

with open(annotation_files[0], "r") as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))
print("Number of image records:", len(data["images"]))
print("Number of annotations:", len(data["annotations"]))

print("\nFirst image record:")
print(data["images"][0])

print("\nFirst annotation record:")
first = data["annotations"][0]
print("Keys:", list(first.keys()))

for key, value in first.items():
    if key == "segmentation":
        print("segmentation:", str(value)[:250], "...")
    else:
        print(key, ":", value)

In [ ]:
import numpy as np

boxes = np.array([
    ann["bbox"]
    for ann in data["annotations"]
    if len(ann["bbox"]) == 4
], dtype=float)

widths = boxes[:, 2]
heights = boxes[:, 3]

valid = (widths > 0) & (heights > 0)
widths = widths[valid]
heights = heights[valid]

aspect_ratios = np.maximum(
    widths / heights,
    heights / widths
)

print("Valid bounding boxes:", len(widths))
print("Median width:", round(np.median(widths), 1))
print("Median height:", round(np.median(heights), 1))
print("Median elongation:", round(np.median(aspect_ratios), 2))
print("90th percentile elongation:",
      round(np.percentile(aspect_ratios, 90), 2))
print("Maximum elongation:", round(aspect_ratios.max(), 2))

In [ ]:
import numpy as np

occupancy = []

for ann in data["annotations"]:
    x, y, w, h = ann["bbox"]
    area = ann["area"]

    if w > 0 and h > 0 and area >= 0:
        occupancy.append(area / (w * h))

occupancy = np.asarray(occupancy)

print("Annotations checked:", len(occupancy))
print("Median occupancy:", round(np.median(occupancy), 3))
print("10th percentile:", round(np.percentile(occupancy, 10), 3))
print("90th percentile:", round(np.percentile(occupancy, 90), 3))
print("Occupancy > 1:", int(np.sum(occupancy > 1)))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import matplotlib.patches as patches

# Find the first training photograph
record = data["images"][0]
filename = record["file_name"]

image_paths = list(Path("/kaggle/input").rglob(filename))
assert image_paths, f"Image not found: {filename}"

image = Image.open(image_paths[0]).convert("L")

# Find annotations belonging to this image record
anns = [
    ann for ann in data["annotations"]
    if ann["image_id"] == record["id"]
]

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(image, cmap="gray")

for ann in anns:
    x, y, w, h = ann["bbox"]

    # Draw bounding box
    rect = patches.Rectangle(
        (x, y), w, h,
        fill=False,
        edgecolor="yellow",
        linewidth=1.5
    )
    ax.add_patch(rect)

    # Draw expert polygon outlines
    for polygon in ann["segmentation"]:
        points = [
            (polygon[i], polygon[i + 1])
            for i in range(0, len(polygon), 2)
        ]
        if len(points) >= 3:
            xs, ys = zip(*(points + [points[0]]))
            ax.plot(xs, ys, color="cyan", linewidth=1)

ax.set_title(
    f"{filename}\n"
    f"{len(anns)} annotated filaments"
)
ax.axis("off")
plt.show()

print("Image size:", image.size)
print("Annotated filaments:", len(anns))

In [ ]:
import numpy as np
from collections import defaultdict

annotations_by_image = defaultdict(list)

for ann in data["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann["bbox"])

def box_iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b

    x1 = max(ax, bx)
    y1 = max(ay, by)
    x2 = min(ax + aw, bx + bw)
    y2 = min(ay + ah, by + bh)

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    union = aw * ah + bw * bh - intersection

    return intersection / union if union > 0 else 0

pairs = 0
overlapping_pairs = 0
high_overlap_pairs = 0

for boxes in annotations_by_image.values():
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            pairs += 1
            iou = box_iou(boxes[i], boxes[j])

            if iou > 0:
                overlapping_pairs += 1
            if iou > 0.3:
                high_overlap_pairs += 1

print("Annotation-record box pairs:", pairs)
print("Overlapping pairs:", overlapping_pairs)
print("Pairs with IoU > 0.3:", high_overlap_pairs)
print("Overlap percentage:",
      round(100 * overlapping_pairs / max(pairs, 1), 2))

In [ ]:
import numpy as np

crop_sizes = []

for ann in data["annotations"]:
    _, _, w, h = ann["bbox"]

    if w > 0 and h > 0:
        crop_w = 1.5 * w
        crop_h = 1.5 * h
        crop_sizes.append(max(crop_w, crop_h))

crop_sizes = np.array(crop_sizes)

print("Filaments:", len(crop_sizes))
print("Median crop side:", round(np.median(crop_sizes), 1))
print("90th percentile:", round(np.percentile(crop_sizes, 90), 1))
print("Maximum:", round(crop_sizes.max(), 1))
print("Crops fitting within 256 pixels:",
      round(100 * np.mean(crop_sizes <= 256), 2), "%")
print("Crops fitting within 512 pixels:",
      round(100 * np.mean(crop_sizes <= 512), 2), "%")

In [ ]:
import numpy as np

oversized = []

for ann in data["annotations"]:
    x, y, w, h = ann["bbox"]

    if w <= 0 or h <= 0:
        continue

    crop_side = 1.5 * max(w, h)

    if crop_side > 512:
        oversized.append({
            "id": ann["id"],
            "image_id": ann["image_id"],
            "crop_side": crop_side
        })

sizes = np.array([a["crop_side"] for a in oversized])

print("Filaments needing more than 512 pixels:", len(oversized))
print("Percentage:", round(100 * len(oversized) / 8199, 2))

if len(sizes):
    print("Median oversized crop:", round(np.median(sizes), 1))
    print("Maximum oversized crop:", round(sizes.max(), 1))
    print("\nFive largest required crops:")
    for a in sorted(
        oversized,
        key=lambda z: z["crop_side"],
        reverse=True
    )[:5]:
        print(a["image_id"], round(a["crop_side"], 1))

In [ ]:
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Largest required crop
largest = max(
    data["annotations"],
    key=lambda a: max(a["bbox"][2], a["bbox"][3])
)

record = next(
    im for im in data["images"]
    if im["id"] == largest["image_id"]
)

image_path = next(
    Path("/kaggle/input").rglob(record["file_name"])
)

image = Image.open(image_path).convert("L")

x, y, w, h = largest["bbox"]
side = int(np.ceil(1.5 * max(w, h)))

cx, cy = x + w / 2, y + h / 2

left = int(round(cx - side / 2))
top = int(round(cy - side / 2))

# PIL pads out-of-image regions with black if necessary
original_crop = image.crop(
    (left, top, left + side, top + side)
)

resized_crop = original_crop.resize(
    (512, 512),
    Image.Resampling.BILINEAR
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(original_crop, cmap="gray")
axes[0].set_title(f"Original crop: {side}×{side}")

axes[1].imshow(resized_crop, cmap="gray")
axes[1].set_title("Resized crop: 512×512")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Image:", record["file_name"])
print("Original crop side:", side)
print("Resize scale:", round(512 / side, 3))

In [ ]:
import numpy as np
import cv2
from PIL import Image, ImageDraw

# Build the original expert mask
mask_image = Image.new("L", (2048, 2048), 0)
draw = ImageDraw.Draw(mask_image)

for polygon in largest["segmentation"]:
    points = [
        (polygon[i], polygon[i + 1])
        for i in range(0, len(polygon), 2)
    ]
    if len(points) >= 3:
        draw.polygon(points, fill=1)

full_mask = np.array(mask_image, dtype=np.uint8)

# Use the exact crop coordinates from Step 8
original_mask = full_mask[
    max(0, top):min(2048, top + side),
    max(0, left):min(2048, left + side)
]

# Resize down, then restore original size
h, w = original_mask.shape

small_mask = cv2.resize(
    original_mask,
    (512, 512),
    interpolation=cv2.INTER_NEAREST
)

restored_mask = cv2.resize(
    small_mask,
    (w, h),
    interpolation=cv2.INTER_NEAREST
)

intersection = np.logical_and(original_mask, restored_mask).sum()
union = np.logical_or(original_mask, restored_mask).sum()

iou = intersection / union if union else 1.0

print("Original foreground pixels:", int(original_mask.sum()))
print("Restored foreground pixels:", int(restored_mask.sum()))
print("Resize reconstruction IoU:", round(iou, 4))

In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path
import numpy as np
import cv2

oversized_anns = [
    ann for ann in data["annotations"]
    if 1.5 * max(ann["bbox"][2:]) > 512
]

rng = np.random.default_rng(42)
indices = rng.choice(
    len(oversized_anns),
    size=min(30, len(oversized_anns)),
    replace=False
)

image_lookup = {im["id"]: im for im in data["images"]}
results = []

for idx in indices:
    ann = oversized_anns[idx]
    record = image_lookup[ann["image_id"]]

    # Build the expert mask
    x, y, w, h = ann["bbox"]
    side = int(np.ceil(1.5 * max(w, h)))
    cx, cy = x + w / 2, y + h / 2

    left = int(round(cx - side / 2))
    top = int(round(cy - side / 2))

    mask_im = Image.new("L", (2048, 2048), 0)
    draw = ImageDraw.Draw(mask_im)

    for polygon in ann["segmentation"]:
        pts = list(zip(polygon[0::2], polygon[1::2]))
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)

    # PIL crop preserves requested size, padding outside the image
    crop = np.asarray(
        mask_im.crop((left, top, left + side, top + side)),
        dtype=np.uint8
    )

    small = cv2.resize(
        crop, (512, 512), interpolation=cv2.INTER_NEAREST
    )

    restored = cv2.resize(
        small, (side, side), interpolation=cv2.INTER_NEAREST
    )

    intersection = np.logical_and(crop, restored).sum()
    union = np.logical_or(crop, restored).sum()

    results.append(intersection / union if union else 1.0)

results = np.asarray(results)

print("Oversized filaments tested:", len(results))
print("Mean reconstruction IoU:", round(results.mean(), 4))
print("Median reconstruction IoU:", round(np.median(results), 4))
print("Minimum reconstruction IoU:", round(results.min(), 4))
print("Filaments with IoU < 0.90:", int(np.sum(results < 0.90)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

# Select an annotation that fits a padded 512×512 crop
candidate = next(
    ann for ann in data["annotations"]
    if 60 <= max(ann["bbox"][2:]) <= 200
    and 1.5 * max(ann["bbox"][2:]) <= 512
)

record = next(
    im for im in data["images"]
    if im["id"] == candidate["image_id"]
)

image_path = next(
    Path("/kaggle/input").rglob(record["file_name"])
)

image = Image.open(image_path).convert("L")

# Create a binary mask for this individual filament only
mask = Image.new("L", image.size, 0)
draw = ImageDraw.Draw(mask)

for polygon in candidate["segmentation"]:
    points = list(zip(polygon[0::2], polygon[1::2]))
    if len(points) >= 3:
        draw.polygon(points, fill=1)

# Centre a 512×512 crop around the filament
x, y, w, h = candidate["bbox"]
cx, cy = x + w / 2, y + h / 2

left = int(round(cx - 256))
top = int(round(cy - 256))

box = (left, top, left + 512, top + 512)

image_crop = np.asarray(image.crop(box))
mask_crop = np.asarray(mask.crop(box))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_crop, cmap="gray")
axes[0].set_title("512×512 image crop")

axes[1].imshow(mask_crop, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Individual filament mask")

axes[2].imshow(image_crop, cmap="gray")
axes[2].imshow(
    np.ma.masked_where(mask_crop == 0, mask_crop),
    cmap="autumn",
    alpha=0.65,
    vmin=0,
    vmax=1
)
axes[2].set_title("Image + target overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Image:", record["file_name"])
print("Crop shape:", image_crop.shape)
print("Mask shape:", mask_crop.shape)
print("Filament pixels:", int(mask_crop.sum()))
print("Mask values:", np.unique(mask_crop))

In [ ]:
# Check whether the selected crop contains other annotated objects
same_image_anns = [
    ann for ann in data["annotations"]
    if ann["image_id"] == candidate["image_id"]
]

overlapping_objects = []

for ann in same_image_anns:
    if ann["id"] == candidate["id"]:
        continue

    x, y, w, h = ann["bbox"]

    intersection_w = max(
        0, min(left + 512, x + w) - max(left, x)
    )
    intersection_h = max(
        0, min(top + 512, y + h) - max(top, y)
    )

    if intersection_w * intersection_h > 0:
        overlapping_objects.append(ann["id"])

print("Target objects:", 1)
print("Other annotation boxes intersecting crop:",
      len(overlapping_objects))
print("Total candidates in crop:",
      1 + len(overlapping_objects))

In [ ]:
from collections import defaultdict

anns_by_image = defaultdict(list)

for ann in data["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

challenging = None

for image_id, anns in anns_by_image.items():
    for target in anns:
        x, y, w, h = target["bbox"]

        if max(w, h) > 300:
            continue

        cx = x + w / 2
        cy = y + h / 2

        left_test = round(cx - 256)
        top_test = round(cy - 256)

        neighbours = []

        for other in anns:
            if other["id"] == target["id"]:
                continue

            ox, oy, ow, oh = other["bbox"]

            overlap_w = max(
                0, min(left_test + 512, ox + ow) - max(left_test, ox)
            )
            overlap_h = max(
                0, min(top_test + 512, oy + oh) - max(top_test, oy)
            )

            if overlap_w * overlap_h > 0:
                neighbours.append(other)

        if neighbours:
            challenging = (image_id, target, neighbours)
            break

    if challenging is not None:
        break

assert challenging is not None, "No multi-object crop found"

image_id, target, neighbours = challenging

print("Image ID:", image_id)
print("Target filament:", target["id"])
print("Other annotation boxes in crop:", len(neighbours))
print("Total annotated candidates:", 1 + len(neighbours))

In [ ]:
from PIL import Image, ImageDraw
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

image_id, target, neighbours = challenging

record = next(
    im for im in data["images"]
    if im["id"] == image_id
)

image_path = next(
    Path("/kaggle/input").rglob(record["file_name"])
)

image = Image.open(image_path).convert("L")

# Centre the crop on our target filament
x, y, w, h = target["bbox"]
cx, cy = x + w / 2, y + h / 2

left = int(round(cx - 256))
top = int(round(cy - 256))
box = (left, top, left + 512, top + 512)

image_crop = np.array(image.crop(box))

def make_instance_crop(ann):
    mask = Image.new("L", image.size, 0)
    draw = ImageDraw.Draw(mask)

    for polygon in ann["segmentation"]:
        points = list(zip(polygon[0::2], polygon[1::2]))
        if len(points) >= 3:
            draw.polygon(points, fill=1)

    return np.array(mask.crop(box), dtype=np.uint8)

target_mask = make_instance_crop(target)

neighbour_mask = np.zeros((512, 512), dtype=np.uint8)
neighbours_with_pixels = 0

for ann in neighbours:
    m = make_instance_crop(ann)
    if m.sum() > 0:
        neighbours_with_pixels += 1
    neighbour_mask |= m

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(image_crop, cmap="gray")
axes[0].set_title("Original solar crop")

axes[1].imshow(image_crop, cmap="gray")

overlay = np.zeros((512, 512, 4), dtype=np.float32)
overlay[neighbour_mask > 0] = [0, 1, 1, 0.75]
overlay[target_mask > 0] = [1, 1, 0, 0.9]

axes[1].imshow(overlay)
axes[1].set_title("Yellow = target | Cyan = neighbours")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Target filament pixels:", int(target_mask.sum()))
print("Neighbours with pixels in crop:", neighbours_with_pixels)
print("Target-neighbour pixel overlap:",
      int(np.logical_and(target_mask, neighbour_mask).sum()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Variables from the previous cell:
# target, left, top, image_crop, target_mask

x, y, w, h = target["bbox"]

# Convert global box coordinates to crop coordinates
x1 = max(0, int(np.floor(x - left)))
y1 = max(0, int(np.floor(y - top)))
x2 = min(512, int(np.ceil(x + w - left)))
y2 = min(512, int(np.ceil(y + h - top)))

box_prompt = np.zeros((512, 512), dtype=np.float32)

if x2 > x1 and y2 > y1:
    box_prompt[y1:y2, x1:x2] = 1.0

# Two-channel input: image + box prompt
model_input = np.stack([
    image_crop.astype(np.float32) / 255.0,
    box_prompt
], axis=0)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(model_input[0], cmap="gray")
axes[0].set_title("Channel 1: Solar image")

axes[1].imshow(box_prompt, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Channel 2: Target box")

axes[2].imshow(target_mask, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Desired output: Target mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Input shape:", model_input.shape)
print("Target shape:", target_mask.shape)
print("Prompt pixels:", int(box_prompt.sum()))
print("Target pixels:", int(target_mask.sum()))

assert model_input.shape == (2, 512, 512)
assert target_mask.shape == (512, 512)
assert np.all(target_mask <= box_prompt)

In [ ]:
import numpy as np

outside = (target_mask > 0) & (box_prompt == 0)

print("Target pixels:", int((target_mask > 0).sum()))
print("Pixels outside prompt:", int(outside.sum()))

if outside.any():
    ys, xs = np.where(outside)

    print("Outside pixel range:")
    print("x:", xs.min(), "to", xs.max())
    print("y:", ys.min(), "to", ys.max())

    print("Prompt bounds:")
    print("x:", x1, "to", x2)
    print("y:", y1, "to", y2)

    print("Fraction of target outside:",
          round(100 * outside.sum() / (target_mask > 0).sum(), 3), "%")

In [ ]:
# Expand the bounding-box prompt by 1 pixel
margin = 1

bx1 = max(0, x1 - margin)
by1 = max(0, y1 - margin)
bx2 = min(512, x2 + margin)
by2 = min(512, y2 + margin)

box_prompt = np.zeros((512, 512), dtype=np.float32)
box_prompt[by1:by2, bx1:bx2] = 1.0

model_input = np.stack([
    image_crop.astype(np.float32) / 255.0,
    box_prompt
], axis=0)

outside = (target_mask > 0) & (box_prompt == 0)

print("Input shape:", model_input.shape)
print("Target pixels:", int(target_mask.sum()))
print("Pixels outside prompt:", int(outside.sum()))

assert model_input.shape == (2, 512, 512)
assert np.all(target_mask <= box_prompt)

print("PASS: Target mask is contained within the box prompt.")

In [ ]:
import torch
import torch.nn as nn

class ConvBlockV6(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class BoxUNetV6(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlockV6(2, 32)
        self.enc2 = ConvBlockV6(32, 64)
        self.enc3 = ConvBlockV6(64, 128)
        self.bottleneck = ConvBlockV6(128, 256)

        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ConvBlockV6(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlockV6(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = ConvBlockV6(64, 32)

        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.output(d1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_v6a = BoxUNetV6().to(device)
model_v6a.eval()

x = torch.from_numpy(model_input).unsqueeze(0).to(device)

with torch.inference_mode():
    logits = model_v6a(x)

print("Device:", device)
print("Input shape:", tuple(x.shape))
print("Output shape:", tuple(logits.shape))
print("Parameters:", sum(p.numel() for p in model_v6a.parameters()))

assert logits.shape == (1, 1, 512, 512)
print("PASS: V6-A forward pass works.")

In [ ]:
from pathlib import Path
import torch

# Locate the saved V5-B checkpoint
matches = list(
    Path("/kaggle/input").rglob("v5b_unet_best.pth")
)

assert len(matches) == 1, f"Found {len(matches)} checkpoints"

checkpoint = torch.load(
    matches[0],
    map_location="cpu",
    weights_only=True
)

# Extract model weights if a training checkpoint wrapper is present
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    old_weights = checkpoint["model_state_dict"]
else:
    old_weights = checkpoint

new_weights = model_v6a.state_dict()

matching = []
different_shape = []
missing = []

for name, tensor in new_weights.items():
    if name not in old_weights:
        missing.append(name)
    elif old_weights[name].shape == tensor.shape:
        matching.append(name)
    else:
        different_shape.append(
            (name, tuple(old_weights[name].shape), tuple(tensor.shape))
        )

print("V5-B checkpoint tensors:", len(old_weights))
print("V6-A model tensors:", len(new_weights))
print("Matching names and shapes:", len(matching))
print("Different shapes:", len(different_shape))
print("Missing names:", len(missing))

print("\nFirst 5 original checkpoint keys:")
for name in list(old_weights.keys())[:5]:
    print(name, tuple(old_weights[name].shape))

print("\nDifferent shapes:")
for item in different_shape[:10]:
    print(item)

print("\nFirst 10 missing names:")
for name in missing[:10]:
    print(name)

In [ ]:
print("V6-A architecture:")
print("  First convolution:", model_v6a.enc1.block[0])
print("  First decoder convolution:", model_v6a.dec3.block[0])
print("  Upsampling:", model_v6a.up3)

print("\nV5-B checkpoint decoder tensors:")
for name, tensor in old_weights.items():
    if name.startswith(("up3", "dec3.block.0")):
        print(name, tuple(tensor.shape))

print("\nV5-B checkpoint first convolution:")
print(
    "enc1.block.0.weight:",
    tuple(old_weights["enc1.block.0.weight"].shape)
)

In [ ]:
print("V5-B checkpoint structure:\n")

for name, tensor in old_weights.items():
    if (
        name.startswith("dec3")
        or name.startswith("dec2")
        or name.startswith("dec1")
        or name.startswith("up")
    ):
        print(name, tuple(tensor.shape))

print("\nFirst convolution:")
print(
    "enc1.block.0.weight",
    tuple(old_weights["enc1.block.0.weight"].shape)
)

In [ ]:
import inspect

print("Original V3 architecture availability:")
print("UNetV3 defined:", "UNetV3" in globals())

if "UNetV3" in globals():
    print(inspect.getsource(UNetV3))
else:
    print("UNetV3 is not defined in this notebook.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Exact convolution block used by V3 / V5-B
class DoubleConvV6A(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class BoxUNetV6A(nn.Module):
    def __init__(self):
        super().__init__()

        # Only architectural modification: 1 -> 2 channels
        self.enc1 = DoubleConvV6A(2, 32)

        self.enc2 = DoubleConvV6A(32, 64)
        self.enc3 = DoubleConvV6A(64, 128)
        self.bottleneck = DoubleConvV6A(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.dec3 = DoubleConvV6A(384, 128)
        self.dec2 = DoubleConvV6A(192, 64)
        self.dec1 = DoubleConvV6A(96, 32)

        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = F.interpolate(
            b, size=e3.shape[2:],
            mode="bilinear", align_corners=False
        )
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = F.interpolate(
            d3, size=e2.shape[2:],
            mode="bilinear", align_corners=False
        )
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = F.interpolate(
            d2, size=e1.shape[2:],
            mode="bilinear", align_corners=False
        )
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.output(d1)


# Build the new model
model_v6a_pretrained = BoxUNetV6A()

# old_weights was loaded from the V5-B checkpoint
# in our earlier compatibility-check cell.
assert "old_weights" in globals(), \
    "Run the earlier V5-B checkpoint inspection cell first."

# Prepare a copy; never modify the original checkpoint
adapted_weights = {
    name: tensor.clone()
    for name, tensor in old_weights.items()
}

first_key = "enc1.block.0.weight"
original_first = adapted_weights[first_key]

assert tuple(original_first.shape) == (32, 1, 3, 3)

new_first = torch.zeros(
    (32, 2, 3, 3),
    dtype=original_first.dtype
)

# Channel 0: original image features
new_first[:, 0:1] = original_first

# Channel 1: box-prompt features start at zero
adapted_weights[first_key] = new_first

# Strict loading verifies compatibility across all layers
model_v6a_pretrained.load_state_dict(
    adapted_weights, strict=True
)

model_v6a_pretrained.eval()

# Test using the actual image and box prompt
x_test = torch.from_numpy(
    model_input
).unsqueeze(0).float()

with torch.inference_mode():
    y_test = model_v6a_pretrained(x_test)

print("Input shape:", tuple(x_test.shape))
print("Output shape:", tuple(y_test.shape))
print("Parameters:", sum(
    p.numel() for p in model_v6a_pretrained.parameters()
))
print("First convolution:",
      tuple(model_v6a_pretrained.enc1.block[0].weight.shape))

print("Original image weights preserved:",
      torch.equal(
          model_v6a_pretrained.enc1.block[0].weight[:, 0:1].cpu(),
          original_first
      ))

print("Prompt weights initialized to zero:",
      torch.count_nonzero(
          model_v6a_pretrained.enc1.block[0].weight[:, 1:2]
      ).item() == 0)

assert y_test.shape == (1, 1, 512, 512)

print("PASS: Pretrained V6-A is ready for dataset preparation.")

In [ ]:
from collections import defaultdict, Counter
from pathlib import Path

# Group annotation records by physical image filename
records_by_file = defaultdict(list)

for record in data["images"]:
    filename = Path(record["file_name"]).name
    records_by_file[filename].append(record["id"])

counts = Counter(
    len(ids) for ids in records_by_file.values()
)

print("Total image records:", len(data["images"]))
print("Unique physical filenames:", len(records_by_file))
print("Annotation records per photograph:")
for n, frequency in sorted(counts.items()):
    print(f"  {n} record(s): {frequency} photographs")

# Count objects associated with each unique photograph
image_id_to_file = {
    record["id"]: Path(record["file_name"]).name
    for record in data["images"]
}

objects_by_file = Counter(
    image_id_to_file[ann["image_id"]]
    for ann in data["annotations"]
)

print("\nTotal individual annotations:", len(data["annotations"]))
print("Photographs with annotations:", len(objects_by_file))

# Check whether we can recover V5-B's original split
possible_split_files = list(
    Path("/kaggle/input").rglob("*split*.json")
) + list(
    Path("/kaggle/input").rglob("*split*.csv")
)

print("\nPotential saved split files:")
for path in possible_split_files[:15]:
    print(path)

if not possible_split_files:
    print("No obvious saved split file found.")

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

# Exact split logic from the original V3/V5-B notebook
filenames = sorted({
    record["file_name"]
    for record in data["images"]
})

train_files, val_files = train_test_split(
    filenames,
    test_size=0.20,
    random_state=42
)

train_set = set(train_files)
val_set = set(val_files)

train_records = [
    record for record in data["images"]
    if record["file_name"] in train_set
]

val_records = [
    record for record in data["images"]
    if record["file_name"] in val_set
]

train_ids = {record["id"] for record in train_records}
val_ids = {record["id"] for record in val_records}

train_annotations = [
    ann for ann in data["annotations"]
    if ann["image_id"] in train_ids
]

val_annotations = [
    ann for ann in data["annotations"]
    if ann["image_id"] in val_ids
]

print("Training photographs:", len(train_files))
print("Validation photographs:", len(val_files))

print("Training annotation records:", len(train_records))
print("Validation annotation records:", len(val_records))

print("Training filament annotations:", len(train_annotations))
print("Validation filament annotations:", len(val_annotations))

print("Shared photographs:", len(train_set & val_set))
print("Shared annotation IDs:", len(train_ids & val_ids))

assert len(train_files) == 565
assert len(val_files) == 142
assert not train_set & val_set
assert not train_ids & val_ids
assert len(train_annotations) + len(val_annotations) == 8199

print("PASS: Original V5-B split reconstructed without leakage.")

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageDraw
from pathlib import Path

class FilamentInstanceDataset(Dataset):
    def __init__(self, annotations, image_records, root, crop_size=512):
        self.annotations = annotations
        self.crop_size = crop_size
        self.root = Path(root)

        self.image_map = {
            record["id"]: record["file_name"]
            for record in image_records
        }

        self.file_map = {
            p.name: p
            for p in self.root.rglob("*.jpeg")
        }

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]

        filename = Path(self.image_map[ann["image_id"]]).name
        image = Image.open(self.file_map[filename]).convert("L")

        W, H = image.size
        S = self.crop_size

        x, y, w, h = ann["bbox"]
        cx, cy = x + w / 2, y + h / 2

        # Keep the crop inside the original photograph
        left = max(0, min(int(round(cx - S / 2)), W - S))
        top = max(0, min(int(round(cy - S / 2)), H - S))

        crop_box = (left, top, left + S, top + S)
        image_crop = np.array(image.crop(crop_box), dtype=np.float32) / 255.0

        # Rasterize the individual filament annotation
        mask = Image.new("L", (W, H), 0)
        draw = ImageDraw.Draw(mask)

        for polygon in ann["segmentation"]:
            points = list(zip(polygon[0::2], polygon[1::2]))
            if len(points) >= 3:
                draw.polygon(points, fill=1)

        target = np.array(mask.crop(crop_box), dtype=np.float32)

        # Bounding-box prompt, with 1-pixel margin
        x1 = max(0, int(np.floor(x - left)) - 1)
        y1 = max(0, int(np.floor(y - top)) - 1)
        x2 = min(S, int(np.ceil(x + w - left)) + 1)
        y2 = min(S, int(np.ceil(y + h - top)) + 1)

        prompt = np.zeros((S, S), dtype=np.float32)
        prompt[y1:y2, x1:x2] = 1.0

        # Two-channel input
        inputs = np.stack([image_crop, prompt], axis=0)

        return (
            torch.from_numpy(inputs),
            torch.from_numpy(target).unsqueeze(0)
        )


train_dataset = FilamentInstanceDataset(
    train_annotations,
    data["images"],
    "/kaggle/input"
)

val_dataset = FilamentInstanceDataset(
    val_annotations,
    data["images"],
    "/kaggle/input"
)

# Inspect one sample
sample_x, sample_y = train_dataset[0]

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Input shape:", tuple(sample_x.shape))
print("Target shape:", tuple(sample_y.shape))
print("Target pixels:", int(sample_y.sum()))
print("Prompt pixels:", int(sample_x[1].sum()))
print("Target pixels outside prompt:",
      int(((sample_y[0] > 0) & (sample_x[1] == 0)).sum()))

assert sample_x.shape == (2, 512, 512)
assert sample_y.shape == (1, 512, 512)

print("PASS: Instance dataset created.")

In [ ]:
import numpy as np
from collections import Counter

def inspect_crop_coverage(annotations, image_records, S=512):
    dimensions = {
        record["id"]: (record["width"], record["height"])
        for record in image_records
    }

    results = Counter()
    examples = []

    for ann in annotations:
        W, H = dimensions[ann["image_id"]]
        x, y, w, h = ann["bbox"]

        cx, cy = x + w / 2, y + h / 2

        left = max(0, min(int(round(cx - S / 2)), W - S))
        top = max(0, min(int(round(cy - S / 2)), H - S))

        right = left + S
        bottom = top + S

        # Does the entire annotated bounding box fit?
        fully_contained = (
            x >= left and y >= top
            and x + w <= right
            and y + h <= bottom
        )

        results["total"] += 1

        if fully_contained:
            results["fully_contained"] += 1
        else:
            results["potentially_truncated"] += 1

            if len(examples) < 5:
                examples.append({
                    "image_id": ann["image_id"],
                    "bbox": ann["bbox"],
                    "crop": (left, top, right, bottom)
                })

    return results, examples


all_annotations = train_annotations + val_annotations

coverage, examples = inspect_crop_coverage(
    all_annotations,
    data["images"]
)

print("Total instances:", coverage["total"])
print("Fully contained bounding boxes:",
      coverage["fully_contained"])
print("Potentially truncated bounding boxes:",
      coverage["potentially_truncated"])

print("Potential truncation rate:",
      round(
          100 * coverage["potentially_truncated"]
          / coverage["total"], 2
      ), "%")

print("\nFirst 5 potentially truncated examples:")
for example in examples:
    print(example)

assert coverage["total"] == 8199

In [ ]:
def is_fully_contained(ann, dimensions, S=512):
    W, H = dimensions[ann["image_id"]]
    x, y, w, h = ann["bbox"]

    cx = x + w / 2
    cy = y + h / 2

    left = max(0, min(int(round(cx - S / 2)), W - S))
    top = max(0, min(int(round(cy - S / 2)), H - S))

    return (
        x >= left
        and y >= top
        and x + w <= left + S
        and y + h <= top + S
    )


dimensions = {
    record["id"]: (record["width"], record["height"])
    for record in data["images"]
}

safe_train_annotations = [
    ann for ann in train_annotations
    if is_fully_contained(ann, dimensions)
]

safe_val_annotations = [
    ann for ann in val_annotations
    if is_fully_contained(ann, dimensions)
]

print("Original training instances:", len(train_annotations))
print("Safe training instances:", len(safe_train_annotations))
print("Excluded training instances:",
      len(train_annotations) - len(safe_train_annotations))

print()
print("Original validation instances:", len(val_annotations))
print("Safe validation instances:", len(safe_val_annotations))
print("Excluded validation instances:",
      len(val_annotations) - len(safe_val_annotations))

assert (
    len(safe_train_annotations) + len(safe_val_annotations)
    == 8119
)

print("\nPASS: Safe instance subsets prepared.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

safe_train_dataset = FilamentInstanceDataset(
    safe_train_annotations,
    data["images"],
    "/kaggle/input"
)

safe_val_dataset = FilamentInstanceDataset(
    safe_val_annotations,
    data["images"],
    "/kaggle/input"
)

# Select reproducible examples
rng = np.random.default_rng(42)
indices = rng.choice(
    len(safe_train_dataset),
    size=4,
    replace=False
)

fig, axes = plt.subplots(4, 3, figsize=(12, 14))

for row, idx in enumerate(indices):
    image_tensor, target_tensor = safe_train_dataset[int(idx)]

    image = image_tensor[0].numpy()
    prompt = image_tensor[1].numpy()
    target = target_tensor[0].numpy()

    axes[row, 0].imshow(image, cmap="gray")
    axes[row, 0].set_title(f"Sample {idx}: Solar image")

    axes[row, 1].imshow(prompt, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Target bounding-box prompt")

    axes[row, 2].imshow(image, cmap="gray")
    overlay = np.zeros((*target.shape, 4))
    overlay[target > 0] = [1, 1, 0, 0.9]
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(
        f"Individual mask ({int(target.sum())} px)"
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

print("Filtered training instances:", len(safe_train_dataset))
print("Filtered validation instances:", len(safe_val_dataset))

for idx in indices:
    x, y = safe_train_dataset[int(idx)]
    outside = ((y[0] > 0) & (x[1] == 0)).sum().item()
    print(
        f"Sample {idx}: "
        f"target={int(y.sum())} px, "
        f"outside prompt={outside} px"
    )

In [ ]:
import torch
import torch.nn.functional as F

def box_instance_loss(logits, targets, prompts):
    """
    logits:  [B, 1, H, W]
    targets: [B, 1, H, W]
    prompts: [B, 1, H, W]
    """

    # Pixel-level supervision
    bce = F.binary_cross_entropy_with_logits(
        logits, targets
    )

    # Foreground-sensitive overlap
    probs = torch.sigmoid(logits)

    intersection = (probs * targets).sum(dim=(1, 2, 3))
    denominator = (
        probs.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = 1 - (
        (2 * intersection + 1)
        / (denominator + 1)
    ).mean()

    # Penalize predictions outside the target box
    outside = 1.0 - prompts

    leakage = (
        (probs * outside).sum(dim=(1, 2, 3))
        / outside.sum(dim=(1, 2, 3)).clamp_min(1)
    ).mean()

    total = bce + dice + 0.5 * leakage

    return total, {
        "BCE": bce.item(),
        "Dice": dice.item(),
        "Leakage": leakage.item()
    }


# Test loss on one dataset sample
sample_x, sample_y = safe_train_dataset[0]

model_v6a_pretrained.eval()

with torch.no_grad():
    logits = model_v6a_pretrained(
        sample_x.unsqueeze(0)
    )

loss, components = box_instance_loss(
    logits,
    sample_y.unsqueeze(0),
    sample_x[1:2].unsqueeze(0)
)

print("Total loss:", round(loss.item(), 6))
print("Loss components:", components)

assert torch.isfinite(loss)
print("PASS: Instance-aware loss is valid.")

In [ ]:
import torch

model = model_v6a_pretrained
model.train()
model.zero_grad(set_to_none=True)

# Use a validated training instance
sample_x, sample_y = safe_train_dataset[0]

inputs = sample_x.unsqueeze(0)
targets = sample_y.unsqueeze(0)
prompts = inputs[:, 1:2]

# Forward pass with gradient tracking
logits = model(inputs)

loss, components = box_instance_loss(
    logits, targets, prompts
)

# Backpropagation only — no optimizer step
loss.backward()

first_layer = model.enc1.block[0]

image_grad = first_layer.weight.grad[:, 0:1]
prompt_grad = first_layer.weight.grad[:, 1:2]

print("Loss:", round(loss.item(), 6))
print("Image-channel gradient norm:",
      image_grad.norm().item())
print("Prompt-channel gradient norm:",
      prompt_grad.norm().item())

print("Prompt gradient finite:",
      torch.isfinite(prompt_grad).all().item())

print("Prompt gradient nonzero:",
      torch.count_nonzero(prompt_grad).item() > 0)

assert torch.isfinite(prompt_grad).all()
assert prompt_grad.norm().item() > 0

print("PASS: The new box-prompt channel can receive gradients.")

# Clear gradients and restore inference mode
model.zero_grad(set_to_none=True)
model.eval()

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB"
    )
else:
    print("No GPU detected. Do not start training yet.")

In [ ]:
import torch
from torch.utils.data import DataLoader

required = [
    "model_v6a_pretrained",
    "safe_train_dataset",
    "safe_val_dataset",
    "box_instance_loss"
]

missing = [name for name in required if name not in globals()]

if missing:
    print("Missing variables:", missing)
    print("Rerun the preparation cells before continuing.")

else:
    device = torch.device("cuda")

    model_v6a_pretrained = model_v6a_pretrained.to(device)
    model_v6a_pretrained.eval()

    pilot_loader = DataLoader(
        safe_train_dataset,
        batch_size=2,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    batch_x, batch_y = next(iter(pilot_loader))

    batch_x = batch_x.to(device)
    batch_y = batch_y.to(device)

    with torch.inference_mode():
        logits = model_v6a_pretrained(batch_x)

        loss, components = box_instance_loss(
            logits,
            batch_y,
            batch_x[:, 1:2]
        )

    print("Device:", device)
    print("Batch input:", tuple(batch_x.shape))
    print("Batch target:", tuple(batch_y.shape))
    print("Model output:", tuple(logits.shape))
    print("Pilot loss:", round(loss.item(), 6))
    print("GPU memory allocated:",
          round(torch.cuda.memory_allocated() / 1024**3, 3),
          "GB")

    assert logits.shape == (2, 1, 512, 512)
    assert torch.isfinite(loss)

    print("PASS: Two-sample GPU batch works.")

In [ ]:
required = [
    "model_v6a_pretrained",
    "safe_train_dataset",
    "safe_val_dataset",
    "box_instance_loss"
]

for name in required:
    print(
        name,
        "READY" if name in globals() else "MISSING"
    )

import torch
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda")

model_v6a_pretrained = model_v6a_pretrained.to(device)
model_v6a_pretrained.eval()

pilot_loader = DataLoader(
    safe_train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

batch_x, batch_y = next(iter(pilot_loader))

batch_x = batch_x.to(device, non_blocking=True)
batch_y = batch_y.to(device, non_blocking=True)

with torch.inference_mode():
    logits = model_v6a_pretrained(batch_x)

    loss, components = box_instance_loss(
        logits,
        batch_y,
        batch_x[:, 1:2]
    )

print("GPU:", torch.cuda.get_device_name(0))
print("Input:", tuple(batch_x.shape))
print("Target:", tuple(batch_y.shape))
print("Output:", tuple(logits.shape))
print("Loss:", round(loss.item(), 6))
print("Loss components:", components)

print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    "GB"
)

assert logits.shape == (2, 1, 512, 512)
assert torch.isfinite(loss)

print("PASS: GPU batch test successful.")

In [ ]:
import copy
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# Reproducible pilot
torch.manual_seed(42)
rng = np.random.default_rng(42)

# Save initial pretrained weights for later comparison
v6a_initial_weights = {
    k: v.detach().cpu().clone()
    for k, v in model_v6a_pretrained.state_dict().items()
}

# Select 128 training instances
pilot_indices = rng.choice(
    len(safe_train_dataset),
    size=128,
    replace=False
).tolist()

train_pilot = DataLoader(
    Subset(safe_train_dataset, pilot_indices),
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

model = model_v6a_pretrained.to("cuda")
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

loss_history = []

step = 0
max_steps = 20

while step < max_steps:
    for images, masks in train_pilot:

        images = images.to("cuda")
        masks = masks.to("cuda")
        prompts = images[:, 1:2]

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)

        loss, components = box_instance_loss(
            logits, masks, prompts
        )

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )
        optimizer.step()

        step += 1
        loss_history.append(loss.item())

        if step == 1 or step % 5 == 0:
            print(
                f"Step {step:02d}/{max_steps} | "
                f"Loss: {loss.item():.4f} | "
                f"BCE: {components['BCE']:.4f} | "
                f"Dice: {components['Dice']:.4f} | "
                f"Leakage: {components['Leakage']:.4f}"
            )

        if step >= max_steps:
            break

model.eval()

print("\nFirst 5-step mean loss:",
      round(np.mean(loss_history[:5]), 5))
print("Last 5-step mean loss:",
      round(np.mean(loss_history[-5:]), 5))

print("PASS: First V6-A pilot training completed.")

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, Subset

# Fixed validation examples
rng = np.random.default_rng(123)
val_indices = rng.choice(
    len(safe_val_dataset), size=12, replace=False
).tolist()

val_loader = DataLoader(
    Subset(safe_val_dataset, val_indices),
    batch_size=2,
    shuffle=False,
    num_workers=0
)

device = torch.device("cuda")

# Original V5-B-initialized model
before_model = BoxUNetV6A().to(device)
before_model.load_state_dict(v6a_initial_weights)
before_model.eval()

# Pilot-trained V6-A model
after_model = model_v6a_pretrained.to(device)
after_model.eval()

def measure_dice(model, loader):
    scores = []

    with torch.inference_mode():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            predictions = (
                torch.sigmoid(model(images)) > 0.5
            ).float()

            intersection = (
                predictions * masks
            ).sum(dim=(1, 2, 3))

            denominator = (
                predictions.sum(dim=(1, 2, 3))
                + masks.sum(dim=(1, 2, 3))
            )

            dice = (
                (2 * intersection + 1e-6)
                / (denominator + 1e-6)
            )

            scores.extend(dice.cpu().tolist())

    return np.array(scores)

before = measure_dice(before_model, val_loader)
after = measure_dice(after_model, val_loader)

print("Validation instances:", len(before))
print("Before training mean Dice:", round(before.mean(), 4))
print("After training mean Dice:", round(after.mean(), 4))
print("Mean Dice change:", round(after.mean() - before.mean(), 4))
print("Instances improved:", int((after > before).sum()), "/ 12")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Use the earlier multi-filament example
image_id, target, neighbours = challenging

# Reuse our existing instance dataset class
test_annotations = [target, neighbours[0]]

test_dataset = FilamentInstanceDataset(
    test_annotations,
    data["images"],
    "/kaggle/input"
)

# Both annotations come from the same solar photograph.
# However, their object-centred crops may differ.
target_x, target_y = test_dataset[0]
neighbour_x, neighbour_y = test_dataset[1]

# Keep the target's image crop fixed for all three conditions.
correct_input = target_x.clone()

# Construct the neighbouring object's box in the
# TARGET crop coordinate system, not the neighbour crop.
x, y, w, h = target["bbox"]

cx = x + w / 2
cy = y + h / 2

W, H = 2048, 2048
left = max(0, min(int(round(cx - 256)), W - 512))
top = max(0, min(int(round(cy - 256)), H - 512))

nx, ny, nw, nh = neighbours[0]["bbox"]

x1 = max(0, int(np.floor(nx - left)) - 1)
y1 = max(0, int(np.floor(ny - top)) - 1)
x2 = min(512, int(np.ceil(nx + nw - left)) + 1)
y2 = min(512, int(np.ceil(ny + nh - top)) + 1)

wrong_input = target_x.clone()
wrong_input[1].zero_()
wrong_input[1, y1:y2, x1:x2] = 1.0

empty_input = target_x.clone()
empty_input[1].zero_()

inputs = torch.stack([
    correct_input,
    wrong_input,
    empty_input
]).to("cuda")

model = model_v6a_pretrained.to("cuda")
model.eval()

with torch.inference_mode():
    probabilities = torch.sigmoid(model(inputs)).cpu().numpy()

predictions = probabilities[:, 0] > 0.5

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

titles = [
    "Original solar crop",
    "Correct box",
    "Neighbour box",
    "Empty box"
]

axes[0, 0].imshow(target_x[0], cmap="gray")
axes[1, 0].imshow(target_y[0], cmap="gray")
axes[1, 0].set_title("Target ground truth")

for j, item in enumerate(
    [correct_input, wrong_input, empty_input], start=1
):
    axes[0, j].imshow(target_x[0], cmap="gray")
    axes[0, j].imshow(item[1], cmap="Reds", alpha=0.4)
    axes[1, j].imshow(predictions[j - 1], cmap="gray")
    axes[1, j].set_title(
        f"Predicted pixels: {predictions[j - 1].sum()}"
    )

for j in range(4):
    axes[0, j].set_title(titles[j])

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

# Compare probability maps, not just thresholded masks.
print(
    "Mean absolute difference: correct vs wrong:",
    round(float(np.abs(probabilities[0] - probabilities[1]).mean()), 6)
)

print(
    "Mean absolute difference: correct vs empty:",
    round(float(np.abs(probabilities[0] - probabilities[2]).mean()), 6)
)

In [ ]:
# Step 40: Quantify target selection and background leakage

import numpy as np

target_gt = target_y[0].numpy() > 0

print("TARGET SELECTION DIAGNOSTIC")
print("-" * 55)

for name, pred, inp in zip(
    ["Correct box", "Neighbour box", "Empty box"],
    predictions,
    [correct_input, wrong_input, empty_input]
):
    box = inp[1].numpy() > 0

    intersection = np.logical_and(pred, target_gt).sum()
    denominator = pred.sum() + target_gt.sum()

    dice = (
        2 * intersection / denominator
        if denominator > 0 else 0
    )

    inside = np.logical_and(pred, box).sum()
    outside = np.logical_and(pred, ~box).sum()

    print(f"\n{name}")
    print("  Dice with target:", round(float(dice), 4))
    print("  Predicted inside box:", int(inside))
    print("  Predicted outside box:", int(outside))
    print(
        "  Fraction outside box:",
        round(float(outside / max(1, pred.sum())), 4)
    )

In [ ]:
import numpy as np

# Reuse the Step 39 predictions and inputs
correct_pred = predictions[0]
wrong_pred = predictions[1]

correct_box = correct_input[1].numpy() > 0
wrong_box = wrong_input[1].numpy() > 0

target_gt = target_y[0].numpy() > 0

# Hard ROI restriction
correct_roi = correct_pred & correct_box
wrong_roi = wrong_pred & wrong_box

def binary_dice(pred, gt):
    intersection = np.logical_and(pred, gt).sum()
    denominator = pred.sum() + gt.sum()

    return 2 * intersection / denominator if denominator else 0.0

print("ROI-CONSTRAINED SEGMENTATION")
print("-" * 45)

print("Original correct-box Dice:",
      round(binary_dice(correct_pred, target_gt), 4))

print("ROI-restricted correct-box Dice:",
      round(binary_dice(correct_roi, target_gt), 4))

print("ROI-restricted wrong-box Dice:",
      round(binary_dice(wrong_roi, target_gt), 4))

print("Correct ROI predicted pixels:", int(correct_roi.sum()))
print("Wrong ROI predicted pixels:", int(wrong_roi.sum()))

print("Target ground-truth pixels:", int(target_gt.sum()))
print("Target pixels covered by correct ROI:",
      int(np.logical_and(correct_roi, target_gt).sum()))

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

rng = np.random.default_rng(2026)

indices = rng.choice(
    len(safe_val_dataset),
    size=32,
    replace=False
).tolist()

loader = DataLoader(
    Subset(safe_val_dataset, indices),
    batch_size=2,
    shuffle=False,
    num_workers=0
)

model_v6a_pretrained.eval()

raw_scores = []
roi_scores = []
roi_recalls = []
roi_precisions = []

with torch.inference_mode():
    for images, masks in loader:
        images = images.to("cuda")
        masks = masks.to("cuda")

        probs = torch.sigmoid(
            model_v6a_pretrained(images)
        )

        raw = probs > 0.5
        boxes = images[:, 1:2] > 0.5
        roi = raw & boxes
        gt = masks > 0.5

        for p_raw, p_roi, target in zip(raw, roi, gt):

            def dice(p, g):
                inter = (p & g).sum().item()
                denom = p.sum().item() + g.sum().item()
                return 2 * inter / denom if denom else 1.0

            intersection = (p_roi & target).sum().item()

            raw_scores.append(dice(p_raw, target))
            roi_scores.append(dice(p_roi, target))

            roi_recalls.append(
                intersection / max(1, target.sum().item())
            )

            roi_precisions.append(
                intersection / max(1, p_roi.sum().item())
            )

raw_scores = np.array(raw_scores)
roi_scores = np.array(roi_scores)

print("Validation instances:", len(raw_scores))
print("Raw mean Dice:", round(raw_scores.mean(), 4))
print("ROI mean Dice:", round(roi_scores.mean(), 4))
print("Mean Dice improvement:",
      round((roi_scores - raw_scores).mean(), 4))

print("ROI mean recall:",
      round(np.mean(roi_recalls), 4))
print("ROI mean precision:",
      round(np.mean(roi_precisions), 4))

print("Instances improved:",
      int((roi_scores > raw_scores).sum()), "/ 32")
print("Instances worsened:",
      int((roi_scores < raw_scores).sum()), "/ 32")

In [ ]:
import torch
import numpy as np
import torch.nn.functional as F

padding_levels = [0, 5, 15, 30]
scores = {p: [] for p in padding_levels}

model_v6a_pretrained.eval()

with torch.inference_mode():
    for images, masks in loader:
        images = images.to("cuda")
        masks = masks.to("cuda")

        gt = masks > 0.5

        for pad in padding_levels:
            original_box = images[:, 1:2]

            # Expand the prompt by 'pad' pixels in each direction
            if pad == 0:
                expanded_box = original_box
            else:
                expanded_box = F.max_pool2d(
                    original_box,
                    kernel_size=2 * pad + 1,
                    stride=1,
                    padding=pad
                )

            modified_input = images.clone()
            modified_input[:, 1:2] = expanded_box

            probabilities = torch.sigmoid(
                model_v6a_pretrained(modified_input)
            )

            pred = (probabilities > 0.5) & (expanded_box > 0.5)

            for p_mask, g_mask in zip(pred, gt):
                intersection = (p_mask & g_mask).sum().item()
                denominator = p_mask.sum().item() + g_mask.sum().item()

                dice = 2 * intersection / denominator if denominator else 1.0
                scores[pad].append(dice)

print("ROI PADDING SENSITIVITY")
print("-" * 40)

for pad in padding_levels:
    print(
        f"Padding {pad:2d} px | "
        f"Mean Dice: {np.mean(scores[pad]):.4f}"
    )

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np

# Use training annotations only; keep validation photographs unseen
train_boxes_by_file = defaultdict(list)

image_info = {
    im["id"]: im for im in data["images"]
}

for ann in safe_train_annotations:
    im = image_info[ann["image_id"]]
    filename = Path(im["file_name"]).name

    x, y, w, h = ann["bbox"]

    if w > 0 and h > 0:
        train_boxes_by_file[filename].append(
            (x, y, w, h)
        )

counts = np.array([
    len(train_boxes_by_file.get(Path(f).name, []))
    for f in train_files
])

widths = np.array([
    box[2]
    for boxes in train_boxes_by_file.values()
    for box in boxes
])

heights = np.array([
    box[3]
    for boxes in train_boxes_by_file.values()
    for box in boxes
])

print("Training photographs:", len(train_files))
print("Photographs with boxes:", int((counts > 0).sum()))
print("Total annotation boxes:", int(counts.sum()))
print("Boxes per photograph — median:", float(np.median(counts)))
print("Boxes per photograph — maximum:", int(counts.max()))

print("Box width — median:", round(float(np.median(widths)), 1))
print("Box height — median:", round(float(np.median(heights)), 1))

print("Boxes smaller than 32 px in both dimensions:",
      int(((widths < 32) & (heights < 32)).sum()))

print("PASS: Detector box statistics prepared.")

In [ ]:
from collections import defaultdict
import numpy as np

# Group boxes by physical photograph and annotation record
groups = defaultdict(lambda: defaultdict(list))

for ann in train_annotations:
    im = image_info[ann["image_id"]]
    filename = im["file_name"]

    groups[filename][ann["image_id"]].append(ann["bbox"])


def bbox_iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b

    ix1 = max(ax, bx)
    iy1 = max(ay, by)
    ix2 = min(ax + aw, bx + bw)
    iy2 = min(ay + ah, by + bh)

    intersection = (
        max(0, ix2 - ix1) *
        max(0, iy2 - iy1)
    )

    union = aw * ah + bw * bh - intersection

    return intersection / union if union > 0 else 0.0


pair_counts = 0
high_overlap = 0
moderate_overlap = 0
examples = []

for filename, records in groups.items():
    record_ids = list(records.keys())

    # Compare only annotations from different records
    for i in range(len(record_ids)):
        for j in range(i + 1, len(record_ids)):

            boxes_a = records[record_ids[i]]
            boxes_b = records[record_ids[j]]

            for a in boxes_a:
                for b in boxes_b:
                    iou = bbox_iou(a, b)
                    pair_counts += 1

                    if iou >= 0.5:
                        high_overlap += 1

                        if len(examples) < 5:
                            examples.append(
                                (filename, round(iou, 3))
                            )

                    if iou >= 0.3:
                        moderate_overlap += 1

print("Cross-annotator box pairs:", pair_counts)
print("Pairs with IoU >= 0.3:", moderate_overlap)
print("Pairs with IoU >= 0.5:", high_overlap)

print("\nExample high-overlap pairs:")
for example in examples:
    print(example)

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np

# Group all original annotations by image record ID
anns_by_record = defaultdict(list)

for ann in data["annotations"]:
    anns_by_record[ann["image_id"]].append(ann)

# Group image records by physical photograph
records_by_photo = defaultdict(list)

for record in data["images"]:
    filename = Path(record["file_name"]).name
    records_by_photo[filename].append(record)

# Select one complete annotation record per photograph
selected_records = {}

for filename, records in records_by_photo.items():
    selected = sorted(
        records,
        key=lambda r: (
            -len(anns_by_record[r["id"]]),
            str(r["id"])
        )
    )[0]

    selected_records[filename] = selected

# Separate by the original physical-image split
detector_train_annotations = []
detector_val_annotations = []

for filename, record in selected_records.items():
    annotations = anns_by_record[record["id"]]

    if record["file_name"] in train_set:
        detector_train_annotations.extend(annotations)

    elif record["file_name"] in val_set:
        detector_val_annotations.extend(annotations)

# Dataset statistics
print("Selected photographs:", len(selected_records))
print("Training photographs:", len(train_set))
print("Validation photographs:", len(val_set))

print("\nDetector training boxes:",
      len(detector_train_annotations))

print("Detector validation boxes:",
      len(detector_val_annotations))

print("Total selected boxes:",
      len(detector_train_annotations)
      + len(detector_val_annotations))

print("\nOriginal annotations:", len(data["annotations"]))

print(
    "Annotations excluded by single-record selection:",
    len(data["annotations"])
    - len(detector_train_annotations)
    - len(detector_val_annotations)
)

# Validate image-record consistency
assert len(selected_records) == 707
assert len(train_set) == 565
assert len(val_set) == 142

assert (
    len(detector_train_annotations)
    + len(detector_val_annotations)
    <= len(data["annotations"])
)

print("\nPASS: Single-expert detector targets prepared.")

In [ ]:
import numpy as np
from collections import defaultdict

# Original image metadata
record_lookup = {
    record["id"]: record
    for record in data["images"]
}

def to_yolo_box(ann):
    record = record_lookup[ann["image_id"]]

    W = record["width"]
    H = record["height"]

    x, y, w, h = ann["bbox"]

    # Convert bounding box to normalized YOLO coordinates
    cx = (x + w / 2) / W
    cy = (y + h / 2) / H
    nw = w / W
    nh = h / H

    return [0, cx, cy, nw, nh]


train_yolo = [
    to_yolo_box(ann)
    for ann in detector_train_annotations
]

val_yolo = [
    to_yolo_box(ann)
    for ann in detector_val_annotations
]

all_boxes = np.array(train_yolo + val_yolo, dtype=np.float64)

# Check class labels, coordinates, dimensions and boundaries
finite = np.isfinite(all_boxes).all(axis=1)

valid = (
    finite
    & (all_boxes[:, 0] == 0)
    & (all_boxes[:, 3] > 0)
    & (all_boxes[:, 4] > 0)
    & (all_boxes[:, 1] - all_boxes[:, 3] / 2 >= -1e-6)
    & (all_boxes[:, 2] - all_boxes[:, 4] / 2 >= -1e-6)
    & (all_boxes[:, 1] + all_boxes[:, 3] / 2 <= 1 + 1e-6)
    & (all_boxes[:, 2] + all_boxes[:, 4] / 2 <= 1 + 1e-6)
)

print("Training YOLO boxes:", len(train_yolo))
print("Validation YOLO boxes:", len(val_yolo))
print("Valid boxes:", int(valid.sum()))
print("Invalid boxes:", int((~valid).sum()))

print("\nFirst 3 YOLO labels:")
for box in train_yolo[:3]:
    print(" ".join(f"{v:.6f}" for v in box))

print("\nNormalized box widths:")
print("Median:", round(float(np.median(all_boxes[:, 3])), 5))
print("Minimum:", round(float(np.min(all_boxes[:, 3])), 5))

print("\nNormalized box heights:")
print("Median:", round(float(np.median(all_boxes[:, 4])), 5))
print("Minimum:", round(float(np.min(all_boxes[:, 4])), 5))

assert len(train_yolo) == 4365
assert len(val_yolo) == 1232

if valid.all():
    print("\nPASS: All YOLO boxes are valid.")
else:
    print("\nCHECK: Some annotations need boundary correction.")
    print("First invalid indices:", np.flatnonzero(~valid)[:10])

In [ ]:
import numpy as np
from collections import defaultdict

TILE = 640
STRIDE = 512

def tile_starts(length, tile=TILE, stride=STRIDE):
    if length <= tile:
        return [0]

    starts = list(range(0, length - tile + 1, stride))

    if starts[-1] != length - tile:
        starts.append(length - tile)

    return starts


def box_inside_tile(box, left, top, tile=TILE):
    x, y, w, h = box

    return (
        x >= left
        and y >= top
        and x + w <= left + tile
        and y + h <= top + tile
    )


# Count whether each detector training annotation
# fits completely inside at least one tile.
covered = 0
uncovered = 0
uncovered_examples = []

tile_counts = {}

for filename in train_files:
    record = selected_records[
        __import__("pathlib").Path(filename).name
    ]

    W = record["width"]
    H = record["height"]

    xs = tile_starts(W)
    ys = tile_starts(H)

    tile_counts[filename] = len(xs) * len(ys)


for ann in detector_train_annotations:
    record = record_lookup[ann["image_id"]]

    W = record["width"]
    H = record["height"]

    xs = tile_starts(W)
    ys = tile_starts(H)

    contained = any(
        box_inside_tile(ann["bbox"], tx, ty)
        for tx in xs
        for ty in ys
    )

    if contained:
        covered += 1
    else:
        uncovered += 1

        if len(uncovered_examples) < 5:
            uncovered_examples.append(
                (ann["image_id"], ann["bbox"])
            )


print("Tile size:", TILE)
print("Stride:", STRIDE)

print("Median tiles per photograph:",
      float(np.median(list(tile_counts.values()))))

print("Total training tiles:",
      sum(tile_counts.values()))

print("\nTraining boxes:", len(detector_train_annotations))
print("Fully covered boxes:", covered)
print("Not fully covered:", uncovered)

print(
    "Coverage:",
    round(100 * covered / len(detector_train_annotations), 2),
    "%"
)

print("\nFirst uncovered examples:")
for example in uncovered_examples:
    print(example)

assert covered + uncovered == len(detector_train_annotations)

In [ ]:
from collections import Counter
import numpy as np

reasons = Counter()
uncovered_sizes = []
examples_by_reason = {}

for ann in detector_train_annotations:
    record = record_lookup[ann["image_id"]]
    W, H = record["width"], record["height"]

    x, y, w, h = ann["bbox"]

    xs = tile_starts(W)
    ys = tile_starts(H)

    covered = any(
        box_inside_tile(ann["bbox"], tx, ty)
        for tx in xs
        for ty in ys
    )

    if covered:
        continue

    if w > TILE or h > TILE:
        reason = "Object larger than 640px"
    else:
        reason = "Object crosses tile boundaries"

    reasons[reason] += 1
    uncovered_sizes.append((w, h))

    if reason not in examples_by_reason:
        examples_by_reason[reason] = (
            record["file_name"],
            ann["bbox"]
        )

print("UNCOVERED FILAMENT ANALYSIS")
print("-" * 45)

for reason, count in reasons.items():
    print(f"{reason}: {count}")

print("\nTotal uncovered:", sum(reasons.values()))

if uncovered_sizes:
    sizes = np.array(uncovered_sizes)

    print("Median uncovered width:",
          round(float(np.median(sizes[:, 0])), 1))

    print("Median uncovered height:",
          round(float(np.median(sizes[:, 1])), 1))

print("\nExample for each reason:")
for reason, example in examples_by_reason.items():
    print(reason, ":", example)

assert sum(reasons.values()) == uncovered

In [ ]:
import numpy as np

def evaluate_tiling(stride):
    covered = 0
    tile_total = 0

    for filename in train_files:
        record = selected_records[Path(filename).name]

        xs = tile_starts(record["width"], 640, stride)
        ys = tile_starts(record["height"], 640, stride)

        tile_total += len(xs) * len(ys)

    for ann in detector_train_annotations:
        record = record_lookup[ann["image_id"]]

        xs = tile_starts(record["width"], 640, stride)
        ys = tile_starts(record["height"], 640, stride)

        if any(
            box_inside_tile(ann["bbox"], tx, ty, 640)
            for tx in xs
            for ty in ys
        ):
            covered += 1

    return covered, tile_total


print("TILING STRATEGY COMPARISON")
print("-" * 65)
print(f"{'Stride':>8} {'Overlap':>9} {'Coverage':>12} {'Tiles':>10}")

for stride in [512, 448, 384, 320, 256]:
    covered, tiles = evaluate_tiling(stride)

    print(
        f"{stride:>8} "
        f"{640 - stride:>9} "
        f"{100 * covered / 4365:>11.2f}% "
        f"{tiles:>10}"
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

TILE = 640
STRIDE = 384

# Select a training image with annotated filaments
filename = Path(train_files[0]).name
record = selected_records[filename]

image_path = next(
    Path("/kaggle/input").rglob(filename)
)

image = Image.open(image_path).convert("L")
W, H = image.size

# Find a tile containing several complete filament boxes
image_annotations = [
    ann for ann in detector_train_annotations
    if ann["image_id"] == record["id"]
]

best_tile = None
best_boxes = []

for top in tile_starts(H, TILE, STRIDE):
    for left in tile_starts(W, TILE, STRIDE):

        contained = [
            ann for ann in image_annotations
            if box_inside_tile(
                ann["bbox"], left, top, TILE
            )
        ]

        if len(contained) > len(best_boxes):
            best_boxes = contained
            best_tile = (left, top)

assert best_tile is not None, "No tile found"

left, top = best_tile

tile_image = np.array(
    image.crop((left, top, left + TILE, top + TILE))
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(tile_image, cmap="gray")

for ann in best_boxes:
    x, y, w, h = ann["bbox"]

    local_x = x - left
    local_y = y - top

    rectangle = plt.Rectangle(
        (local_x, local_y),
        w, h,
        fill=False,
        edgecolor="yellow",
        linewidth=1.5
    )

    ax.add_patch(rectangle)

ax.set_title(
    f"YOLOX training tile — {len(best_boxes)} complete boxes"
)
ax.axis("off")
plt.show()

print("Image:", filename)
print("Original dimensions:", (W, H))
print("Tile origin:", (left, top))
print("Tile shape:", tile_image.shape)
print("Fully contained boxes:", len(best_boxes))

assert tile_image.shape == (640, 640)
assert all(
    0 <= ann["bbox"][0] - left
    and 0 <= ann["bbox"][1] - top
    and ann["bbox"][0] + ann["bbox"][2] - left <= TILE
    and ann["bbox"][1] + ann["bbox"][3] - top <= TILE
    for ann in best_boxes
)

print("PASS: Tile-local bounding boxes are valid.")

In [ ]:
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np

TILE = 640
STRIDE = 384

# Group the selected detector annotations by image record
detector_anns_by_id = defaultdict(list)

for ann in detector_train_annotations:
    detector_anns_by_id[ann["image_id"]].append(ann)

tile_stats = Counter()
example_partial_tiles = []

for filename in train_files:
    record = selected_records[Path(filename).name]

    anns = detector_anns_by_id[record["id"]]

    for top in tile_starts(record["height"], TILE, STRIDE):
        for left in tile_starts(record["width"], TILE, STRIDE):

            complete = 0
            partial = 0

            for ann in anns:
                x, y, w, h = ann["bbox"]

                ix = max(
                    0, min(left + TILE, x + w) - max(left, x)
                )
                iy = max(
                    0, min(top + TILE, y + h) - max(top, y)
                )

                if ix * iy == 0:
                    continue

                if box_inside_tile(
                    ann["bbox"], left, top, TILE
                ):
                    complete += 1
                else:
                    partial += 1

            if complete > 0:
                tile_stats["positive"] += 1
            elif partial > 0:
                tile_stats["partial_only"] += 1

                if len(example_partial_tiles) < 3:
                    example_partial_tiles.append(
                        (Path(filename).name, left, top)
                    )
            else:
                tile_stats["no_intersecting_boxes"] += 1

            tile_stats["total"] += 1

print("DETECTOR TILE DISTRIBUTION")
print("-" * 45)

print("Total tiles:", tile_stats["total"])
print("Tiles with complete filaments:",
      tile_stats["positive"])
print("Tiles containing only partial boxes:",
      tile_stats["partial_only"])
print("Tiles without intersecting boxes:",
      tile_stats["no_intersecting_boxes"])

print("\nExamples of partial-only tiles:")
for example in example_partial_tiles:
    print(example)

assert tile_stats["total"] == 14125
assert (
    tile_stats["positive"]
    + tile_stats["partial_only"]
    + tile_stats["no_intersecting_boxes"]
    == tile_stats["total"]
)

print("\nPASS: Tile distribution classified.")

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path

clean_stats = Counter()
clean_tiles = []

for filename in train_files:
    record = selected_records[Path(filename).name]
    anns = detector_anns_by_id[record["id"]]

    for top in tile_starts(record["height"], 640, 384):
        for left in tile_starts(record["width"], 640, 384):

            complete = []
            partial = []

            for ann in anns:
                x, y, w, h = ann["bbox"]

                iw = max(
                    0, min(left + 640, x + w) - max(left, x)
                )
                ih = max(
                    0, min(top + 640, y + h) - max(top, y)
                )

                if iw * ih <= 0:
                    continue

                if box_inside_tile(ann["bbox"], left, top, 640):
                    complete.append(ann)
                else:
                    partial.append(ann)

            if partial:
                clean_stats["excluded_partial"] += 1
                if complete:
                    clean_stats["excluded_mixed"] += 1
            elif complete:
                clean_stats["clean_positive"] += 1
                clean_tiles.append(
                    (Path(filename).name, left, top, complete)
                )
            else:
                clean_stats["clean_negative"] += 1

print("CLEAN DETECTOR TILE ANALYSIS")
print("-" * 45)

for key in [
    "clean_positive",
    "clean_negative",
    "excluded_partial",
    "excluded_mixed"
]:
    print(f"{key}: {clean_stats[key]}")

print("\nTotal retained tiles:",
      clean_stats["clean_positive"]
      + clean_stats["clean_negative"])

print("Total clean-positive boxes:",
      sum(len(t[3]) for t in clean_tiles))

assert (
    clean_stats["clean_positive"]
    + clean_stats["clean_negative"]
    + clean_stats["excluded_partial"]
    == 14125
)

print("PASS: Clean detector tile candidates identified.")

In [ ]:
import numpy as np
from collections import defaultdict
from pathlib import Path

rng = np.random.default_rng(2026)

# Clean positive tiles from Step 53
positive_tiles = clean_tiles

# Reconstruct clean negative tiles
negative_tiles = []

for filename in train_files:
    record = selected_records[Path(filename).name]
    anns = detector_anns_by_id[record["id"]]

    for top in tile_starts(record["height"], 640, 384):
        for left in tile_starts(record["width"], 640, 384):

            has_intersection = False

            for ann in anns:
                x, y, w, h = ann["bbox"]

                iw = max(
                    0, min(left + 640, x + w) - max(left, x)
                )
                ih = max(
                    0, min(top + 640, y + h) - max(top, y)
                )

                if iw * ih > 0:
                    has_intersection = True
                    break

            if not has_intersection:
                negative_tiles.append(
                    (Path(filename).name, left, top, [])
                )

# Select reproducible subsets
pos_indices = rng.choice(
    len(positive_tiles), size=200, replace=False
)

neg_indices = rng.choice(
    len(negative_tiles), size=100, replace=False
)

detector_pilot_tiles = (
    [positive_tiles[int(i)] for i in pos_indices]
    + [negative_tiles[int(i)] for i in neg_indices]
)

rng.shuffle(detector_pilot_tiles)

positive_count = sum(
    len(tile[3]) > 0 for tile in detector_pilot_tiles
)

negative_count = sum(
    len(tile[3]) == 0 for tile in detector_pilot_tiles
)

print("Pilot training tiles:", len(detector_pilot_tiles))
print("Positive tiles:", positive_count)
print("Negative tiles:", negative_count)
print("Total positive-box occurrences:",
      sum(len(tile[3]) for tile in detector_pilot_tiles))

print("Unique source photographs:",
      len({tile[0] for tile in detector_pilot_tiles}))

assert len(detector_pilot_tiles) == 300
assert positive_count == 200
assert negative_count == 100

print("PASS: Detector pilot tiles selected.")

In [ ]:
import json
from pathlib import Path
from PIL import Image
from collections import Counter

output_root = Path("/kaggle/working/v6a_detector_pilot")
image_dir = output_root / "train2017"
annotation_dir = output_root / "annotations"

image_dir.mkdir(parents=True, exist_ok=True)
annotation_dir.mkdir(parents=True, exist_ok=True)

coco = {
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "filament", "supercategory": "solar"}
    ]
}

# Locate source photographs once
source_files = {
    p.name: p
    for p in Path("/kaggle/input").rglob("*.jpeg")
}

assert all(
    filename in source_files
    for filename, _, _, _ in detector_pilot_tiles
), "Some source photographs could not be located."

annotation_id = 1

for image_id, (filename, left, top, anns) in enumerate(
    detector_pilot_tiles, start=1
):
    output_name = f"tile_{image_id:05d}.jpg"

    with Image.open(source_files[filename]) as source:
        tile = source.convert("RGB").crop(
            (left, top, left + 640, top + 640)
        )
        tile.save(
            image_dir / output_name,
            quality=95
        )

    coco["images"].append({
        "id": image_id,
        "file_name": output_name,
        "width": 640,
        "height": 640
    })

    for ann in anns:
        x, y, w, h = ann["bbox"]

        local_x = float(x - left)
        local_y = float(y - top)
        w = float(w)
        h = float(h)

        # The selected tiles should contain complete boxes
        assert local_x >= -1e-5
        assert local_y >= -1e-5
        assert local_x + w <= 640 + 1e-5
        assert local_y + h <= 640 + 1e-5

        coco["annotations"].append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": 1,
            "bbox": [local_x, local_y, w, h],
            "area": w * h,
            "iscrowd": 0
        })

        annotation_id += 1

json_path = annotation_dir / "train.json"

with open(json_path, "w") as f:
    json.dump(coco, f)

# Read the exported file back to verify it
with open(json_path) as f:
    exported = json.load(f)

counts = Counter(
    ann["image_id"]
    for ann in exported["annotations"]
)

positive = sum(
    counts[im["id"]] > 0
    for im in exported["images"]
)

negative = sum(
    counts[im["id"]] == 0
    for im in exported["images"]
)

print("Export directory:", output_root)
print("Exported images:", len(exported["images"]))
print("Exported annotations:", len(exported["annotations"]))
print("Positive images:", positive)
print("Negative images:", negative)

print("Image files on disk:",
      len(list(image_dir.glob("*.jpg"))))

print("COCO JSON exists:", json_path.exists())

assert len(exported["images"]) == 300
assert len(exported["annotations"]) == 300
assert positive == 200
assert negative == 100
assert len(list(image_dir.glob("*.jpg"))) == 300

print("PASS: YOLOX pilot dataset exported successfully.")

In [ ]:
from collections import defaultdict, Counter
from pathlib import Path

TILE = 640
STRIDE = 384

# Group selected validation annotations by image record
val_anns_by_id = defaultdict(list)

for ann in detector_val_annotations:
    val_anns_by_id[ann["image_id"]].append(ann)

val_clean_positive = []
val_clean_negative = []
val_tile_stats = Counter()

for filename in val_files:
    record = selected_records[Path(filename).name]
    anns = val_anns_by_id[record["id"]]

    for top in tile_starts(record["height"], TILE, STRIDE):
        for left in tile_starts(record["width"], TILE, STRIDE):

            complete = []
            partial = []

            for ann in anns:
                x, y, w, h = ann["bbox"]

                iw = max(
                    0,
                    min(left + TILE, x + w) - max(left, x)
                )
                ih = max(
                    0,
                    min(top + TILE, y + h) - max(top, y)
                )

                if iw * ih <= 0:
                    continue

                if box_inside_tile(
                    ann["bbox"], left, top, TILE
                ):
                    complete.append(ann)
                else:
                    partial.append(ann)

            if partial:
                val_tile_stats["excluded_partial"] += 1

            elif complete:
                val_clean_positive.append(
                    (Path(filename).name, left, top, complete)
                )
                val_tile_stats["clean_positive"] += 1

            else:
                val_clean_negative.append(
                    (Path(filename).name, left, top, [])
                )
                val_tile_stats["clean_negative"] += 1

            val_tile_stats["total"] += 1

print("VALIDATION TILE DISTRIBUTION")
print("-" * 45)

print("Validation photographs:", len(val_files))
print("Total validation tiles:", val_tile_stats["total"])
print("Clean positive tiles:", val_tile_stats["clean_positive"])
print("Clean negative tiles:", val_tile_stats["clean_negative"])
print("Excluded partial tiles:", val_tile_stats["excluded_partial"])

print(
    "Total retained validation tiles:",
    len(val_clean_positive) + len(val_clean_negative)
)

print(
    "Complete-box occurrences in positive tiles:",
    sum(len(tile[3]) for tile in val_clean_positive)
)

assert (
    val_tile_stats["clean_positive"]
    + val_tile_stats["clean_negative"]
    + val_tile_stats["excluded_partial"]
    == val_tile_stats["total"]
)

assert set(train_files).isdisjoint(set(val_files))

print("\nPASS: Held-out validation tile candidates prepared.")

In [ ]:
import json
from pathlib import Path
from PIL import Image
from collections import Counter

root = Path("/kaggle/working/v6a_detector_pilot")
val_dir = root / "val2017"
ann_dir = root / "annotations"

val_dir.mkdir(parents=True, exist_ok=True)
ann_dir.mkdir(parents=True, exist_ok=True)

validation_tiles = val_clean_positive + val_clean_negative

val_coco = {
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "filament", "supercategory": "solar"}
    ]
}

annotation_id = 1

for image_id, (filename, left, top, anns) in enumerate(
    validation_tiles, start=1
):
    output_name = f"val_tile_{image_id:05d}.jpg"

    with Image.open(source_files[filename]) as source:
        tile = source.convert("RGB").crop(
            (left, top, left + 640, top + 640)
        )
        tile.save(val_dir / output_name, quality=95)

    val_coco["images"].append({
        "id": image_id,
        "file_name": output_name,
        "width": 640,
        "height": 640
    })

    for ann in anns:
        x, y, w, h = ann["bbox"]

        local_x = float(x - left)
        local_y = float(y - top)

        assert local_x >= -1e-5
        assert local_y >= -1e-5
        assert local_x + w <= 640 + 1e-5
        assert local_y + h <= 640 + 1e-5

        val_coco["annotations"].append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": 1,
            "bbox": [
                local_x, local_y, float(w), float(h)
            ],
            "area": float(w * h),
            "iscrowd": 0
        })

        annotation_id += 1

val_json = ann_dir / "val.json"

with open(val_json, "w") as f:
    json.dump(val_coco, f)

# Verify the exported dataset
with open(val_json) as f:
    exported_val = json.load(f)

counts = Counter(
    ann["image_id"]
    for ann in exported_val["annotations"]
)

positive = sum(
    counts[im["id"]] > 0
    for im in exported_val["images"]
)

negative = sum(
    counts[im["id"]] == 0
    for im in exported_val["images"]
)

print("Validation images:", len(exported_val["images"]))
print("Validation annotations:", len(exported_val["annotations"]))
print("Positive tiles:", positive)
print("Negative tiles:", negative)
print("Image files saved:", len(list(val_dir.glob("*.jpg"))))
print("COCO JSON:", val_json)

assert len(exported_val["images"]) == 2038
assert len(exported_val["annotations"]) == 1269
assert positive == 791
assert negative == 1247
assert len(list(val_dir.glob("*.jpg"))) == 2038

print("PASS: Held-out COCO validation dataset exported.")

In [ ]:
from pathlib import Path
import json

root = Path("/kaggle/working/v6a_detector_pilot")
ann_dir = root / "annotations"

try:
    from pycocotools.coco import COCO
    coco_available = True
except ImportError:
    coco_available = False

print("pycocotools available:", coco_available)

if coco_available:
    for split, expected_images, expected_anns in [
        ("train", 300, 300),
        ("val", 2038, 1269)
    ]:
        path = ann_dir / f"{split}.json"
        coco_api = COCO(str(path))

        image_ids = coco_api.getImgIds()
        category_ids = coco_api.getCatIds()
        annotation_ids = coco_api.getAnnIds()

        print(f"\n{split.upper()} DATASET")
        print("Images:", len(image_ids))
        print("Annotations:", len(annotation_ids))
        print("Category IDs:", category_ids)

        assert len(image_ids) == expected_images
        assert len(annotation_ids) == expected_anns
        assert category_ids == [1]

        image_dir = root / (
            "train2017" if split == "train" else "val2017"
        )

        missing_files = [
            im["file_name"]
            for im in coco_api.loadImgs(image_ids)
            if not (image_dir / im["file_name"]).exists()
        ]

        print("Missing image files:", len(missing_files))
        assert not missing_files

    print("\nPASS: Both datasets are COCO-compatible.")

else:
    print(
        "COCO API is not installed. "
        "We will install the required dependency "
        "before configuring YOLOX."
    )

In [ ]:
import importlib.util
import subprocess
import sys
import torch

packages = [
    "yolox",
    "torchvision",
    "pycocotools",
    "cv2",
    "loguru",
    "tabulate",
    "thop"
]

print("DETECTOR ENVIRONMENT CHECK")
print("-" * 45)

for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(f"{package:15s}: {'AVAILABLE' if available else 'MISSING'}")

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# Check for a previously installed YOLOX command
result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "yolox"],
    capture_output=True,
    text=True
)

print("\nYOLOX pip installation:")
print(
    result.stdout[:1000]
    if result.returncode == 0
    else "Not installed through pip"
)

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/kaggle/working/YOLOX")

if not (repo / "setup.py").exists():
    result = subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Megvii-BaseDetection/YOLOX.git",
            str(repo)
        ],
        capture_output=True,
        text=True,
        timeout=120
    )

    print("Clone return code:", result.returncode)

    if result.returncode != 0:
        print("Clone error:", result.stderr[-1200:])

else:
    print("YOLOX source already present.")

if (repo / "setup.py").exists():
    print("\nPASS: Official YOLOX source available.")

    print("\nAvailable lightweight configurations:")
    for path in sorted((repo / "exps").rglob("*nano*.py")):
        print(path.relative_to(repo))
    for path in sorted((repo / "exps").rglob("*tiny*.py")):
        print(path.relative_to(repo))

    print("\nDeclared dependencies:")
    requirements = repo / "requirements.txt"
    if requirements.exists():
        print(requirements.read_text()[:3000])

    result = subprocess.run(
        ["git", "-C", str(repo), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True
    )
    print("\nRepository commit:", result.stdout.strip())

In [ ]:
import sys
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/YOLOX")

assert (repo / "setup.py").exists(), "YOLOX source is missing."

# Install only missing lightweight dependencies.
# Do not reinstall or downgrade PyTorch.
required = ["loguru", "thop", "ninja", "tensorboard"]

import importlib.util

missing = [
    package for package in required
    if importlib.util.find_spec(package) is None
]

if missing:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", *missing],
        capture_output=True,
        text=True
    )

    print("Dependency installation return code:",
          result.returncode)

    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("Dependency installation failed.")
else:
    print("Essential dependencies already available.")

# Load the cloned source directly, without compiling extensions
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
from yolox.exp import get_exp

print("PASS: Core YOLOX modules imported.")

# Load the official Tiny experiment configuration
exp = get_exp(
    exp_file=str(repo / "exps/default/yolox_tiny.py")
)

print("Experiment:", exp.exp_name)
print("Default input size:", exp.input_size)
print("Depth multiplier:", exp.depth)
print("Width multiplier:", exp.width)

# Build a one-class model; do not train yet
exp.num_classes = 1
detector_model = exp.get_model()

parameters = sum(
    p.numel() for p in detector_model.parameters()
)

print("Detector classes:", exp.num_classes)
print("Model parameters:", f"{parameters:,}")

assert exp.num_classes == 1
print("PASS: Official YOLOX-Tiny model constructed.")

In [ ]:
import torch
from PIL import Image
import numpy as np

# Configure YOLOX for our native-resolution tiles
exp.input_size = (640, 640)
exp.test_size = (640, 640)
exp.num_classes = 1

detector_model = detector_model.to("cuda")
detector_model.eval()

# Use one real exported solar tile
tile_path = sorted(
    (output_root / "train2017").glob("*.jpg")
)[0]

image = np.array(
    Image.open(tile_path).convert("RGB"),
    dtype=np.float32
)

assert image.shape == (640, 640, 3)

# YOLOX expects B × C × H × W
# Use raw 0–255 pixel intensities for this architecture
input_tensor = torch.from_numpy(
    image.transpose(2, 0, 1).copy()
).unsqueeze(0).to("cuda")

with torch.inference_mode():
    output = detector_model(input_tensor)

print("GPU:", torch.cuda.get_device_name(0))
print("Input shape:", tuple(input_tensor.shape))
print("Output shape:", tuple(output.shape))
print("Output finite:", torch.isfinite(output).all().item())

print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    "GB"
)

assert output.ndim == 3
assert output.shape[0] == 1
assert output.shape[2] == 6
assert torch.isfinite(output).all()

print("PASS: YOLOX-Tiny 640×640 GPU forward pass successful.")

In [ ]:
import torch

detector_model = detector_model.to("cuda")
detector_model.train()

# Synthetic training targets with realistic box dimensions.
# YOLOX target format:
# [class_id, center_x, center_y, width, height]
# Coordinates are in pixels, not normalized.

images = input_tensor.repeat(2, 1, 1, 1).contiguous()

targets = torch.zeros(
    (2, 5, 5),
    dtype=torch.float32,
    device="cuda"
)

# First image: two filament boxes
targets[0, 0] = torch.tensor(
    [0, 200, 250, 75, 65], device="cuda"
)
targets[0, 1] = torch.tensor(
    [0, 400, 350, 110, 45], device="cuda"
)

# Second image: one filament box
targets[1, 0] = torch.tensor(
    [0, 300, 300, 80, 70], device="cuda"
)

print("Image batch:", tuple(images.shape))
print("Target batch:", tuple(targets.shape))

try:
    with torch.enable_grad():
        results = detector_model(images, targets)

    if isinstance(results, dict):
        for key, value in results.items():
            if torch.is_tensor(value):
                print(f"{key}: {value.item():.6f}")
            else:
                print(f"{key}: {value}")

        assert torch.isfinite(results["total_loss"])

    print("PASS: YOLOX training loss works.")

except Exception as error:
    print("Training loss test failed.")
    print("Error type:", type(error).__name__)
    print("Error:", str(error)[:1500])

finally:
    detector_model.eval()

In [ ]:
import requests
from pathlib import Path
import torch

checkpoint_url = (
    "https://github.com/Megvii-BaseDetection/YOLOX/"
    "releases/download/0.1.1rc0/yolox_tiny.pth"
)

checkpoint_path = Path(
    "/kaggle/working/yolox_tiny_pretrained.pth"
)

if not checkpoint_path.exists():
    try:
        response = requests.get(
            checkpoint_url,
            timeout=60,
            stream=True,
            allow_redirects=True
        )
        response.raise_for_status()

        with open(checkpoint_path, "wb") as f:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)

        print("Checkpoint downloaded.")

    except Exception as error:
        print("Download failed:", str(error)[:500])

if checkpoint_path.exists():
    print(
        "Checkpoint size:",
        round(checkpoint_path.stat().st_size / 1024**2, 2),
        "MB"
    )

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        weights = (
            checkpoint["model"]
            if isinstance(checkpoint, dict)
            and "model" in checkpoint
            else checkpoint
        )

        print("Checkpoint tensors:", len(weights))
        print("First three keys:", list(weights)[:3])

        print(
            "PASS: Pretrained YOLOX-Tiny checkpoint readable."
        )

    except Exception as error:
        print("Checkpoint loading failed:", str(error)[:600])

In [ ]:
import torch

# Original pretrained checkpoint from Step 64
pretrained_weights = weights

# Our one-class YOLOX-Tiny model
detector_weights = detector_model.state_dict()

compatible = {}
incompatible = []

for name, tensor in pretrained_weights.items():

    if name in detector_weights:
        if tensor.shape == detector_weights[name].shape:
            compatible[name] = tensor
        else:
            incompatible.append(
                (
                    name,
                    tuple(tensor.shape),
                    tuple(detector_weights[name].shape)
                )
            )

# Transfer matching pretrained parameters
load_result = detector_model.load_state_dict(
    compatible,
    strict=False
)

print("PRETRAINED WEIGHT TRANSFER")
print("-" * 50)

print("Checkpoint tensors:", len(pretrained_weights))
print("Transferred tensors:", len(compatible))
print("Shape-incompatible tensors:", len(incompatible))

print("\nIncompatible layers:")
for item in incompatible:
    print(item)

print("\nMissing model keys:", len(load_result.missing_keys))
print("Unexpected keys:", len(load_result.unexpected_keys))

# Verify that only classification-head tensors differ
assert compatible, "No compatible pretrained weights found."

assert all(
    "cls_preds" in name
    for name, _, _ in incompatible
), "Unexpected architectural incompatibility detected."

assert set(load_result.missing_keys) == {
    name for name, _, _ in incompatible
}, "Unexpected missing parameters."

detector_model = detector_model.to("cuda")
detector_model.eval()

print("\nPASS: Pretrained YOLOX-Tiny initialized for one-class detection.")

In [ ]:
import json
import numpy as np
import torch
from PIL import Image
from pathlib import Path

root = Path("/kaggle/working/v6a_detector_pilot")

with open(root / "annotations/train.json") as f:
    coco_train = json.load(f)

# Group annotations by image ID
from collections import defaultdict

anns_by_tile = defaultdict(list)

for ann in coco_train["annotations"]:
    anns_by_tile[ann["image_id"]].append(ann)

# Choose two real positive training tiles
positive_images = [
    im for im in coco_train["images"]
    if len(anns_by_tile[im["id"]]) > 0
]

selected = positive_images[:2]

images = []
targets = []

for im in selected:
    path = root / "train2017" / im["file_name"]

    image = np.asarray(
        Image.open(path).convert("RGB"),
        dtype=np.float32
    )

    image = np.ascontiguousarray(
        image.transpose(2, 0, 1)
    )

    # YOLOX expects pixel-space CX, CY, W, H
    boxes = torch.zeros((50, 5), dtype=torch.float32)

    tile_anns = anns_by_tile[im["id"]]
    assert len(tile_anns) <= 50

    for j, ann in enumerate(tile_anns):
        x, y, w, h = ann["bbox"]

        boxes[j] = torch.tensor([
            0,             # Filament class
            x + w / 2,     # Center X
            y + h / 2,     # Center Y
            w,
            h
        ])

    images.append(torch.from_numpy(image))
    targets.append(boxes)

batch_images = torch.stack(images).to("cuda")
batch_targets = torch.stack(targets).to("cuda")

detector_model.train()

try:
    results = detector_model(
        batch_images,
        batch_targets
    )

    print("Image batch:", tuple(batch_images.shape))
    print("Target batch:", tuple(batch_targets.shape))

    for name, value in results.items():
        if torch.is_tensor(value):
            print(f"{name}: {value.item():.6f}")

    assert torch.isfinite(results["total_loss"])

    print("PASS: Real solar annotations work with YOLOX.")

finally:
    detector_model.eval()

In [ ]:
import json
import random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

root = Path("/kaggle/working/v6a_detector_pilot")

class SolarYOLOXDataset(Dataset):
    def __init__(self, root):
        self.root = Path(root)

        with open(self.root / "annotations/train.json") as f:
            coco = json.load(f)

        self.images = coco["images"]
        self.anns = {im["id"]: [] for im in self.images}

        for ann in coco["annotations"]:
            self.anns[ann["image_id"]].append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        im = self.images[idx]

        path = self.root / "train2017" / im["file_name"]

        image = np.asarray(
            Image.open(path).convert("RGB"),
            dtype=np.float32
        )

        image = torch.from_numpy(
            np.ascontiguousarray(image.transpose(2, 0, 1))
        )

        targets = torch.zeros((50, 5), dtype=torch.float32)

        anns = self.anns[im["id"]]
        assert len(anns) <= 50

        for j, ann in enumerate(anns):
            x, y, w, h = ann["bbox"]

            targets[j] = torch.tensor([
                0, x + w / 2, y + h / 2, w, h
            ])

        return image, targets


dataset = SolarYOLOXDataset(root)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

detector_model = detector_model.to("cuda")
detector_model.train()

# Keep a separate copy of the pretrained starting point
detector_initial_state = {
    k: v.detach().cpu().clone()
    for k, v in detector_model.state_dict().items()
}

optimizer = torch.optim.AdamW(
    detector_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

loss_history_detector = []
steps = 0
max_steps = 20

while steps < max_steps:
    for images, targets in loader:
        images = images.to("cuda", non_blocking=True)
        targets = targets.to("cuda", non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        results = detector_model(images, targets)
        loss = results["total_loss"]

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite YOLOX loss")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            detector_model.parameters(), 1.0
        )
        optimizer.step()

        steps += 1
        loss_history_detector.append(float(loss.item()))

        if steps == 1 or steps % 5 == 0:
            print(
                f"Step {steps:02d}/{max_steps} | "
                f"Total: {loss.item():.4f} | "
                f"IoU: {results['iou_loss'].item():.4f} | "
                f"Objectness: {results['conf_loss'].item():.4f} | "
                f"Class: {results['cls_loss'].item():.4f}"
            )

        if steps >= max_steps:
            break

detector_model.eval()

# Save separately from the original YOLOX checkpoint
save_path = Path(
    "/kaggle/working/v6a_yolox_tiny_pilot20.pth"
)

torch.save(
    {
        "model": detector_model.state_dict(),
        "steps": steps,
        "loss_history": loss_history_detector,
        "num_classes": 1,
        "input_size": (640, 640)
    },
    save_path
)

print("\nFirst 5-step mean loss:",
      round(np.mean(loss_history_detector[:5]), 4))
print("Last 5-step mean loss:",
      round(np.mean(loss_history_detector[-5:]), 4))

print("Checkpoint saved:", save_path)
print("PASS: YOLOX training pilot completed.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

from yolox.utils import postprocess

root = Path("/kaggle/working/v6a_detector_pilot")

# Select a positive validation tile
val_images = exported_val["images"]
val_annotations = exported_val["annotations"]

val_counts = {}
for ann in val_annotations:
    image_id = ann["image_id"]
    val_counts[image_id] = val_counts.get(image_id, 0) + 1

# Prefer a tile containing at least 2 annotated filaments
sample_info = next(
    im for im in val_images
    if val_counts.get(im["id"], 0) >= 2
)

image_path = root / "val2017" / sample_info["file_name"]

image = np.array(
    Image.open(image_path).convert("RGB"),
    dtype=np.float32
)

x = torch.from_numpy(
    np.ascontiguousarray(image.transpose(2, 0, 1))
).unsqueeze(0).to("cuda")

detector_model.eval()

with torch.inference_mode():
    raw_output = detector_model(x)

    detections = postprocess(
        raw_output.clone(),
        num_classes=1,
        conf_thre=0.05,
        nms_thre=0.5,
        class_agnostic=True
    )[0]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image.astype(np.uint8))

# Ground-truth boxes in green
gt_anns = [
    ann for ann in val_annotations
    if ann["image_id"] == sample_info["id"]
]

for ann in gt_anns:
    gx, gy, gw, gh = ann["bbox"]
    ax.add_patch(
        plt.Rectangle(
            (gx, gy), gw, gh,
            fill=False, edgecolor="lime",
            linewidth=2
        )
    )

# Predicted boxes in red
num_detections = 0

if detections is not None:
    det = detections.cpu().numpy()
    num_detections = len(det)

    for row in det:
        x1, y1, x2, y2 = row[:4]
        confidence = float(row[4] * row[5])

        ax.add_patch(
            plt.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor="red",
                linewidth=1.5
            )
        )

        ax.text(
            x1, max(0, y1 - 3),
            f"{confidence:.2f}",
            color="red", fontsize=8
        )

ax.set_title("Green = expert boxes | Red = YOLOX predictions")
ax.axis("off")
plt.tight_layout()
plt.show()

print("Validation image:", sample_info["file_name"])
print("Ground-truth boxes:", len(gt_anns))
print("Predicted boxes:", num_detections)
print("Confidence threshold:", 0.05)
print("NMS IoU threshold:", 0.5)

if detections is not None:
    print(
        "Maximum predicted confidence:",
        round(float((detections[:, 4] * detections[:, 5]).max()), 5)
    )
else:
    print("No detections passed the confidence threshold.")

In [ ]:
import torch
import numpy as np

detector_model.eval()

with torch.inference_mode():
    raw = detector_model(x)

# YOLOX output:
# [center_x, center_y, width, height, objectness, class_score]

objectness = raw[0, :, 4]
class_scores = raw[0, :, 5]
confidence = objectness * class_scores

print("YOLOX CONFIDENCE DIAGNOSTIC")
print("-" * 45)

print("Total prediction candidates:", confidence.numel())
print("Maximum objectness:", round(objectness.max().item(), 6))
print("Maximum class score:", round(class_scores.max().item(), 6))
print("Maximum combined confidence:", round(confidence.max().item(), 6))

for threshold in [0.001, 0.005, 0.01, 0.02, 0.05]:
    count = (confidence >= threshold).sum().item()
    print(f"Candidates >= {threshold:.3f}: {count}")

# Examine the highest-confidence candidates
top_scores, top_indices = torch.topk(confidence, k=5)

print("\nTop 5 candidates:")
for score, index in zip(top_scores, top_indices):
    box = raw[0, index, :4].cpu().numpy()

    print(
        f"Confidence={score.item():.6f} | "
        f"Box (cx,cy,w,h)={np.round(box, 1)}"
    )

assert torch.isfinite(raw).all()
print("\nPASS: Confidence diagnostic completed.")

In [ ]:
import numpy as np
import torch

# Raw predictions from Step 69
raw_np = raw[0].detach().cpu().numpy()

scores = raw_np[:, 4] * raw_np[:, 5]

# Expert boxes from the same validation tile
gt_boxes = np.array(
    [ann["bbox"] for ann in gt_anns],
    dtype=np.float32
)

def xywh_to_xyxy(box):
    x, y, w, h = box
    return np.array([x, y, x + w, y + h])

def box_iou_xyxy(a, b):
    ix1 = max(a[0], b[0])
    iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2])
    iy2 = min(a[3], b[3])

    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)

    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0

# YOLOX inference boxes use cx, cy, width, height
def prediction_to_xyxy(box):
    cx, cy, w, h = box
    return np.array([
        cx - w / 2,
        cy - h / 2,
        cx + w / 2,
        cy + h / 2
    ])

gt_xyxy = [xywh_to_xyxy(b) for b in gt_boxes]

print("TOP-CANDIDATE LOCALIZATION")
print("-" * 50)

for k in [5, 20, 100, 500]:
    top_indices = np.argsort(scores)[::-1][:k]

    predicted_boxes = [
        prediction_to_xyxy(raw_np[i, :4])
        for i in top_indices
    ]

    best_ious = []

    for gt in gt_xyxy:
        best = max(
            box_iou_xyxy(pred, gt)
            for pred in predicted_boxes
        )
        best_ious.append(best)

    print(
        f"Top {k:3d} candidates | "
        f"Best IoUs per GT: "
        f"{[round(v, 3) for v in best_ious]} | "
        f"GT covered @ IoU 0.5: "
        f"{sum(v >= 0.5 for v in best_ious)}/{len(gt_xyxy)}"
    )

In [ ]:
import torch
import numpy as np
from pathlib import Path

# Continue from the existing 20-step pilot
detector_model = detector_model.to("cuda")
detector_model.train()

# Fresh optimizer with conservative learning rate
optimizer = torch.optim.AdamW(
    detector_model.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

additional_steps = 80
step = 0
extension_losses = []

while step < additional_steps:

    for images, targets in loader:
        images = images.to("cuda", non_blocking=True)
        targets = targets.to("cuda", non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        results = detector_model(images, targets)
        loss = results["total_loss"]

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite YOLOX training loss")

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            detector_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        step += 1
        extension_losses.append(loss.item())

        if step == 1 or step % 20 == 0:
            print(
                f"Total step {20 + step:03d}/100 | "
                f"Loss: {loss.item():.4f} | "
                f"IoU: {results['iou_loss'].item():.4f} | "
                f"Objectness: {results['conf_loss'].item():.4f}"
            )

        if step >= additional_steps:
            break

detector_model.eval()

save_path = Path(
    "/kaggle/working/v6a_yolox_tiny_pilot100.pth"
)

torch.save({
    "model": detector_model.state_dict(),
    "steps": 100,
    "loss_history": loss_history_detector + extension_losses,
    "num_classes": 1,
    "input_size": (640, 640)
}, save_path)

print("\nExtension first 10-step mean:",
      round(np.mean(extension_losses[:10]), 4))
print("Extension last 10-step mean:",
      round(np.mean(extension_losses[-10:]), 4))

print("Checkpoint:", save_path)
print("PASS: YOLOX 100-step pilot completed.")

In [ ]:
import torch
import numpy as np
from pathlib import Path
from yolox.utils import postprocess

# Load the saved 20-step detector
ckpt20 = torch.load(
    "/kaggle/working/v6a_yolox_tiny_pilot20.pth",
    map_location="cpu",
    weights_only=True
)

model20 = exp.get_model().to("cuda")
model20.load_state_dict(ckpt20["model"], strict=True)
model20.eval()

# Current 100-step detector
model100 = detector_model.to("cuda")
model100.eval()

# Same validation image and GT boxes from Steps 68–70
assert x.shape == (1, 3, 640, 640)
assert len(gt_xyxy) > 0

def evaluate_detector(model, image_tensor, ground_truth):
    with torch.inference_mode():
        raw_output = model(image_tensor)

        confidence = (
            raw_output[0, :, 4]
            * raw_output[0, :, 5]
        )

        detections = postprocess(
            raw_output.clone(),
            num_classes=1,
            conf_thre=0.001,
            nms_thre=0.5,
            class_agnostic=True
        )[0]

    # Diagnostic top-100 candidate localization
    raw_np = raw_output[0].cpu().numpy()
    scores = confidence.cpu().numpy()

    indices = np.argsort(scores)[::-1][:100]

    boxes = [
        prediction_to_xyxy(raw_np[i, :4])
        for i in indices
    ]

    best_ious = [
        max(box_iou_xyxy(box, gt) for box in boxes)
        for gt in ground_truth
    ]

    n_detections = (
        0 if detections is None else len(detections)
    )

    return {
        "max_conf": float(confidence.max().item()),
        "above_005": int((confidence >= 0.05).sum().item()),
        "nms_001": n_detections,
        "best_ious": best_ious
    }

results20 = evaluate_detector(model20, x, gt_xyxy)
results100 = evaluate_detector(model100, x, gt_xyxy)

print("YOLOX VALIDATION COMPARISON")
print("-" * 55)

for label, result in [
    ("20 steps", results20),
    ("100 steps", results100)
]:
    print(f"\n{label}")
    print("Maximum confidence:",
          round(result["max_conf"], 6))
    print("Candidates >= 0.05:",
          result["above_005"])
    print("Detections after NMS @ 0.001:",
          result["nms_001"])
    print(
        "Top-100 best IoUs:",
        [round(v, 3) for v in result["best_ious"]]
    )

print("\nPASS: Controlled detector comparison completed.")

In [ ]:
import torch

print("MODEL INDEPENDENCE CHECK")
print("-" * 50)

print("Same Python model object:",
      model20 is model100)

p20 = next(model20.parameters())
p100 = next(model100.parameters())

print("Same parameter memory:",
      p20.data_ptr() == p100.data_ptr())

print("First parameter identical:",
      torch.equal(p20, p100))

# Compare the actual saved checkpoints directly,
# without loading either into a model.
ckpt100 = torch.load(
    "/kaggle/working/v6a_yolox_tiny_pilot100.pth",
    map_location="cpu",
    weights_only=True
)

state20 = ckpt20["model"]
state100 = ckpt100["model"]

changed = []
max_differences = []

for name in state20:
    a = state20[name]
    b = state100[name]

    if torch.is_floating_point(a):
        diff = (a - b).abs().max().item()
        if diff > 0:
            changed.append(name)
            max_differences.append(diff)
    elif not torch.equal(a, b):
        changed.append(name)

print("\nSaved checkpoint comparison:")
print("20-step tensors:", len(state20))
print("100-step tensors:", len(state100))
print("Changed tensors:", len(changed))

if max_differences:
    print("Largest absolute weight difference:",
          max(max_differences))

print("Example changed tensor names:")
for name in changed[:5]:
    print(" ", name)

In [ ]:
import copy
import torch

# Create genuinely independent model objects
model20 = copy.deepcopy(detector_model).to("cuda")
model100 = copy.deepcopy(detector_model).to("cuda")

# Restore their respective saved checkpoints
model20.load_state_dict(ckpt20["model"], strict=True)
model100.load_state_dict(ckpt100["model"], strict=True)

model20.eval()
model100.eval()

# Verify independence
assert model20 is not model100

p20 = next(model20.parameters())
p100 = next(model100.parameters())

assert p20.data_ptr() != p100.data_ptr()

print("Independent models:", model20 is not model100)
print("Separate parameter memory:", p20.data_ptr() != p100.data_ptr())

# Reuse the same held-out validation tile and evaluation function
results20 = evaluate_detector(model20, x, gt_xyxy)
results100 = evaluate_detector(model100, x, gt_xyxy)

print("\nCORRECTED YOLOX COMPARISON")
print("-" * 50)

for label, result in [
    ("20 steps", results20),
    ("100 steps", results100)
]:
    print(f"\n{label}")
    print("Maximum confidence:",
          round(result["max_conf"], 6))
    print("Candidates >= 0.05:", result["above_005"])
    print("Detections after NMS @ 0.001:", result["nms_001"])
    print(
        "Top-100 best IoUs:",
        [round(float(v), 3) for v in result["best_ious"]]
    )

print("\nPASS: Independent checkpoint comparison completed.")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from yolox.utils import postprocess

model100.eval()

with torch.inference_mode():
    output100 = model100(x)

    detections100 = postprocess(
        output100.clone(),
        num_classes=1,
        conf_thre=0.05,
        nms_thre=0.5,
        class_agnostic=True
    )[0]

det = (
    detections100.cpu().numpy()
    if detections100 is not None
    else np.empty((0, 7))
)

pred_boxes = det[:, :4]
pred_scores = det[:, 4] * det[:, 5]

# Match each ground-truth filament to its best NMS prediction
print("100-STEP YOLOX — FINAL DETECTIONS")
print("-" * 50)

print("Ground-truth boxes:", len(gt_xyxy))
print("Predicted boxes after NMS:", len(pred_boxes))

for i, gt in enumerate(gt_xyxy):
    overlaps = [
        box_iou_xyxy(pred, gt)
        for pred in pred_boxes
    ]

    if overlaps:
        best_index = int(np.argmax(overlaps))
        print(
            f"GT {i + 1}: "
            f"best IoU={overlaps[best_index]:.3f}, "
            f"confidence={pred_scores[best_index]:.4f}"
        )
    else:
        print(f"GT {i + 1}: no detection")

# Visualize expert versus predicted boxes
fig, ax = plt.subplots(figsize=(8, 8))

ax.imshow(image.astype(np.uint8))

for gt in gt_xyxy:
    x1, y1, x2, y2 = gt
    ax.add_patch(
        plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="lime",
            linewidth=2
        )
    )

for box, score in zip(pred_boxes, pred_scores):
    x1, y1, x2, y2 = box

    ax.add_patch(
        plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="red",
            linewidth=1.5
        )
    )

    ax.text(
        x1, max(0, y1 - 3),
        f"{score:.2f}",
        color="red",
        fontsize=8
    )

ax.set_title(
    "100-step YOLOX | Green: GT | Red: predicted"
)
ax.axis("off")
plt.tight_layout()
plt.show()

print("PASS: Post-NMS detection inspection completed.")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Same held-out tile from Step 75
solar_gray = image[:, :, 0].astype(np.float32) / 255.0

# Select the best predicted box for each GT filament
matched_boxes = []

for gt in gt_xyxy:
    overlaps = np.array([
        box_iou_xyxy(pred, gt)
        for pred in pred_boxes
    ])

    best_idx = int(np.argmax(overlaps))
    matched_boxes.append(pred_boxes[best_idx])

# Construct 512x512 instance crops centred on detected boxes
# Keep each crop inside the 640x640 tile
instance_inputs = []
crop_origins = []

for box in matched_boxes:
    x1, y1, x2, y2 = box

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    left = int(np.clip(round(cx - 256), 0, 128))
    top = int(np.clip(round(cy - 256), 0, 128))

    crop = solar_gray[
        top:top + 512,
        left:left + 512
    ]

    prompt = np.zeros((512, 512), dtype=np.float32)

    bx1 = int(np.clip(np.floor(x1 - left) - 1, 0, 512))
    by1 = int(np.clip(np.floor(y1 - top) - 1, 0, 512))
    bx2 = int(np.clip(np.ceil(x2 - left) + 1, 0, 512))
    by2 = int(np.clip(np.ceil(y2 - top) + 1, 0, 512))

    prompt[by1:by2, bx1:bx2] = 1.0

    instance_inputs.append(
        np.stack([crop, prompt], axis=0)
    )
    crop_origins.append((left, top))

batch = torch.from_numpy(
    np.stack(instance_inputs)
).float().to("cuda")

segmenter = model_v6a_pretrained.to("cuda")
segmenter.eval()

with torch.inference_mode():
    probabilities = torch.sigmoid(segmenter(batch))

    # Restrict output to the predicted bounding box
    masks = (
        (probabilities > 0.5)
        & (batch[:, 1:2] > 0.5)
    ).cpu().numpy()

# Restore masks to the original 640x640 tile coordinates
full_masks = []

for i, (left, top) in enumerate(crop_origins):
    full = np.zeros((640, 640), dtype=np.uint8)

    full[
        top:top + 512,
        left:left + 512
    ] = masks[i, 0].astype(np.uint8)

    full_masks.append(full)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(solar_gray, cmap="gray")
axes[0].set_title("Original solar tile")

for i, mask in enumerate(full_masks):
    axes[i + 1].imshow(solar_gray, cmap="gray")

    overlay = np.zeros((640, 640, 4))
    overlay[mask > 0] = [1, 1, 0, 0.9]

    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(
        f"Detected filament {i + 1}: {int(mask.sum())} pixels"
    )

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Detector boxes used:", len(matched_boxes))
print("Instance masks produced:", len(full_masks))
print("Mask pixel counts:", [int(m.sum()) for m in full_masks])

assert len(full_masks) == 2
assert all(m.shape == (640, 640) for m in full_masks)

print("PASS: YOLOX predictions connected to V6-A segmentation.")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Detections from Step 75
assert len(pred_boxes) > 0, "No detector predictions available"

# The same held-out 640x640 solar tile
gray = image[:, :, 0].astype(np.float32) / 255.0

instance_inputs = []
crop_origins = []
valid_scores = []

for box, score in zip(pred_boxes, pred_scores):
    x1, y1, x2, y2 = map(float, box)

    # Centre a 512x512 crop on the detected object
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    left = int(np.clip(round(cx - 256), 0, 128))
    top = int(np.clip(round(cy - 256), 0, 128))

    crop = gray[top:top+512, left:left+512]

    # Create box-prompt channel
    prompt = np.zeros((512, 512), dtype=np.float32)

    bx1 = int(np.clip(np.floor(x1-left)-1, 0, 512))
    by1 = int(np.clip(np.floor(y1-top)-1, 0, 512))
    bx2 = int(np.clip(np.ceil(x2-left)+1, 0, 512))
    by2 = int(np.clip(np.ceil(y2-top)+1, 0, 512))

    if bx2 <= bx1 or by2 <= by1:
        continue

    prompt[by1:by2, bx1:bx2] = 1.0

    instance_inputs.append(
        np.stack([crop, prompt], axis=0)
    )
    crop_origins.append((left, top))
    valid_scores.append(float(score))

assert instance_inputs, "No valid detection prompts"

# Batch inference
segmenter = model_v6a_pretrained.to("cuda")
segmenter.eval()

batch = torch.from_numpy(
    np.stack(instance_inputs)
).float().to("cuda")

with torch.inference_mode():
    probabilities = torch.sigmoid(segmenter(batch))

    local_masks = (
        (probabilities > 0.5)
        & (batch[:, 1:2] > 0.5)
    ).cpu().numpy()

# Project masks back into 640x640 tile coordinates
automatic_masks = []

for i, (left, top) in enumerate(crop_origins):
    full_mask = np.zeros((640, 640), dtype=np.uint8)

    full_mask[top:top+512, left:left+512] = (
        local_masks[i, 0].astype(np.uint8)
    )

    automatic_masks.append(full_mask)

# Visualization: separate colours for different instances
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(gray, cmap="gray")

colours = plt.cm.tab10(np.linspace(0, 1, 10))

for i, mask in enumerate(automatic_masks):
    overlay = np.zeros((640, 640, 4), dtype=np.float32)
    overlay[mask > 0, :3] = colours[i % 10, :3]
    overlay[mask > 0, 3] = 0.75
    ax.imshow(overlay)

ax.set_title("Automatic YOLOX → V6-A instance masks")
ax.axis("off")
plt.show()

print("YOLOX detections:", len(pred_boxes))
print("Generated instance masks:", len(automatic_masks))
print("Mask areas:", [int(m.sum()) for m in automatic_masks])
print("Detector confidences:",
      [round(s, 3) for s in valid_scores])

print("PASS: Automatic detector-to-mask inference completed.")

In [ ]:
import numpy as np

# Use automatic masks and detector scores from Step 77
masks = np.asarray(automatic_masks, dtype=bool)
scores = np.asarray(valid_scores, dtype=float)

areas = masks.reshape(len(masks), -1).sum(axis=1)

# Remove empty masks
valid = np.flatnonzero(areas > 0)

masks_valid = masks[valid]
scores_valid = scores[valid]

n = len(valid)
iou_matrix = np.zeros((n, n), dtype=np.float32)

for i in range(n):
    for j in range(n):
        intersection = np.logical_and(
            masks_valid[i], masks_valid[j]
        ).sum()

        union = np.logical_or(
            masks_valid[i], masks_valid[j]
        ).sum()

        iou_matrix[i, j] = (
            intersection / union if union else 0.0
        )

print("MASK OVERLAP DIAGNOSTIC")
print("-" * 45)

print("Original masks:", len(masks))
print("Nonempty masks:", n)
print("Original indices:", valid.tolist())

print("\nPairwise mask IoU:")
print(np.round(iou_matrix, 3))

print("\nOverlapping pairs:")

for i in range(n):
    for j in range(i + 1, n):
        if iou_matrix[i, j] > 0:
            print(
                f"Masks {valid[i]} and {valid[j]} | "
                f"IoU={iou_matrix[i, j]:.3f} | "
                f"Scores={scores_valid[i]:.3f}, "
                f"{scores_valid[j]:.3f}"
            )

assert n == 6

print("\nPASS: Mask overlap analysis completed.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MASK_NMS_THRESHOLD = 0.50

# Nonempty masks and scores from Step 78
order = np.argsort(-scores_valid)

keep = []
suppressed = []

while len(order) > 0:
    current = int(order[0])
    keep.append(current)

    remaining = order[1:]
    survivors = []

    for j in remaining:
        j = int(j)

        if iou_matrix[current, j] > MASK_NMS_THRESHOLD:
            suppressed.append(
                (int(valid[j]), int(valid[current]),
                 float(iou_matrix[current, j]))
            )
        else:
            survivors.append(j)

    order = np.array(survivors, dtype=int)

kept_original_indices = valid[keep]
final_masks = masks_valid[keep]
final_scores = scores_valid[keep]

print("MASK NMS RESULTS")
print("-" * 45)
print("Nonempty input masks:", len(masks_valid))
print("Masks retained:", len(final_masks))
print("Original indices retained:", kept_original_indices.tolist())

print("\nSuppressed masks (removed, kept, IoU):")
for item in suppressed:
    print(item)

print("\nRetained mask areas:",
      [int(m.sum()) for m in final_masks])

print("Retained confidences:",
      [round(float(s), 3) for s in final_scores])

# Visualize retained instances
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(gray, cmap="gray")

colors = plt.cm.tab10(np.linspace(0, 1, 10))

for i, mask in enumerate(final_masks):
    overlay = np.zeros((640, 640, 4), dtype=np.float32)
    overlay[mask, :3] = colors[i % 10, :3]
    overlay[mask, 3] = 0.75
    ax.imshow(overlay)

ax.set_title(
    f"Automatic instance masks after Mask NMS: {len(final_masks)}"
)
ax.axis("off")
plt.show()

assert len(final_masks) <= len(masks_valid)
print("PASS: Greedy Mask NMS completed.")

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment

# Rasterize the expert polygons into 640x640 masks
ground_truth_masks = []

for ann in gt_anns:
    mask = Image.new("L", (640, 640), 0)
    draw = ImageDraw.Draw(mask)

    # These annotations were exported as boxes only,
    # so recover their original polygons by image ID
    original = next(
        a for a in detector_val_annotations
        if (
            a["image_id"] == selected_records[
                next(
                    name for name, r in selected_records.items()
                    if r["id"] == record_lookup[
                        a["image_id"]
                    ]["id"]
                )
            ]["id"]
            and abs(a["bbox"][2] - ann["bbox"][2]) < 1e-4
        )
    )

    # Original polygons are in full-image coordinates.
    # Tile origin must be recovered before drawing.
    ground_truth_masks.append(mask)

print("GT masks prepared:", len(ground_truth_masks))

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

# sample_info comes from Step 68.
# validation_tiles was used to export val_tile_00001.jpg, etc.
tile_index = int(
    sample_info["file_name"].replace("val_tile_", "").replace(".jpg", "")
) - 1

filename, tile_left, tile_top, original_anns = validation_tiles[tile_index]

assert len(original_anns) == len(gt_anns)

ground_truth_masks = []

for ann in original_anns:
    mask = Image.new("L", (640, 640), 0)
    draw = ImageDraw.Draw(mask)

    for polygon in ann["segmentation"]:
        points = list(zip(
            [v - tile_left for v in polygon[0::2]],
            [v - tile_top for v in polygon[1::2]]
        ))

        if len(points) >= 3:
            draw.polygon(points, fill=1)

    ground_truth_masks.append(
        np.asarray(mask, dtype=bool)
    )

gt_masks = np.stack(ground_truth_masks)
pred_masks = np.asarray(final_masks, dtype=bool)

iou = np.zeros(
    (len(pred_masks), len(gt_masks)),
    dtype=np.float32
)

for i, pred in enumerate(pred_masks):
    for j, gt in enumerate(gt_masks):
        intersection = np.logical_and(pred, gt).sum()
        union = np.logical_or(pred, gt).sum()
        iou[i, j] = intersection / union if union else 0

print("Predicted masks:", len(pred_masks))
print("Ground-truth masks:", len(gt_masks))
print("\nMask IoU matrix:")
print(np.round(iou, 3))

from scipy.optimize import linear_sum_assignment

rows, cols = linear_sum_assignment(-iou)

matches = [
    (int(r), int(c), float(iou[r, c]))
    for r, c in zip(rows, cols)
    if iou[r, c] >= 0.5
]

tp = len(matches)
fp = len(pred_masks) - tp
fn = len(gt_masks) - tp

print("\nMatches (prediction, GT, IoU):", matches)
print("TP:", tp, "| FP:", fp, "| FN:", fn)

print(
    "Mean matched IoU:",
    round(np.mean([m[2] for m in matches]), 4)
    if matches else 0.0
)

print("PASS: Instance-matching diagnostic completed.")

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def evaluate_mask_nms(threshold):
    order = np.argsort(-scores_valid)
    keep = []

    while len(order):
        current = int(order[0])
        keep.append(current)

        order = np.array([
            int(j) for j in order[1:]
            if iou_matrix[current, int(j)] <= threshold
        ], dtype=int)

    # Use the precomputed prediction-vs-GT IoUs
    # The original 'iou' matrix corresponds to final_masks
    # from Step 79, so recompute from masks_valid.
    kept_masks = masks_valid[keep]

    matrix = np.zeros(
        (len(kept_masks), len(gt_masks)),
        dtype=np.float32
    )

    for i, pred in enumerate(kept_masks):
        for j, gt in enumerate(gt_masks):
            inter = np.logical_and(pred, gt).sum()
            union = np.logical_or(pred, gt).sum()
            matrix[i, j] = inter / union if union else 0.0

    rows, cols = linear_sum_assignment(-matrix)

    matched_ious = [
        float(matrix[r, c])
        for r, c in zip(rows, cols)
        if matrix[r, c] >= 0.5
    ]

    tp = len(matched_ious)
    fp = len(kept_masks) - tp
    fn = len(gt_masks) - tp

    sq = np.mean(matched_ious) if tp else 0.0

    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = sum(matched_ious) / denominator if denominator else 0.0

    return len(kept_masks), tp, fp, fn, sq, pq


print("MASK NMS THRESHOLD COMPARISON")
print("-" * 66)
print("Threshold | Masks | TP | FP | FN |   SQ   | Tile PQ")

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    n, tp, fp, fn, sq, pq = evaluate_mask_nms(threshold)

    print(
        f"   {threshold:.1f}    |"
        f"  {n:2d}   |"
        f" {tp:2d} |"
        f" {fp:2d} |"
        f" {fn:2d} |"
        f" {sq:.4f} |"
        f" {pq:.4f}"
    )

print("\nPASS: Mask NMS sensitivity evaluated.")

In [ ]:
import numpy as np
from collections import Counter

# Validation COCO data from Step 57
val_images = exported_val["images"]
val_anns = exported_val["annotations"]

counts = Counter(
    ann["image_id"] for ann in val_anns
)

# Select tiles containing at least one annotated filament
positive_val_images = [
    im for im in val_images
    if counts[im["id"]] > 0
]

rng = np.random.default_rng(2026)

selected_indices = rng.choice(
    len(positive_val_images),
    size=20,
    replace=False
)

evaluation_tiles = [
    positive_val_images[int(i)]
    for i in selected_indices
]

print("MULTI-TILE EVALUATION SET")
print("-" * 45)

print("Available positive validation tiles:",
      len(positive_val_images))

print("Selected evaluation tiles:",
      len(evaluation_tiles))

print("Total ground-truth box occurrences:",
      sum(counts[im["id"]] for im in evaluation_tiles))

print("\nFirst five selected tiles:")
for im in evaluation_tiles[:5]:
    print(
        im["file_name"],
        "| Filaments:",
        counts[im["id"]]
    )

assert len(evaluation_tiles) == 20

print("\nPASS: Multi-tile evaluation subset selected.")

In [ ]:
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from yolox.utils import postprocess

device = "cuda"
root = Path("/kaggle/working/v6a_detector_pilot")

detector = model100.to(device).eval()
segmenter = model_v6a_pretrained.to(device).eval()

evaluation_predictions = {}

with torch.inference_mode():

    for index, info in enumerate(evaluation_tiles, start=1):

        filename = info["file_name"]

        rgb = np.array(
            Image.open(root / "val2017" / filename).convert("RGB"),
            dtype=np.float32
        )

        gray = rgb[:, :, 0] / 255.0

        detector_input = torch.from_numpy(
            np.ascontiguousarray(rgb.transpose(2, 0, 1))
        ).unsqueeze(0).to(device)

        raw_output = detector(detector_input)

        detections = postprocess(
            raw_output.clone(),
            num_classes=1,
            conf_thre=0.05,
            nms_thre=0.5,
            class_agnostic=True
        )[0]

        masks = []
        scores = []

        if detections is not None:

            det = detections.cpu().numpy()

            prompts_and_crops = []
            crop_origins = []
            candidate_scores = []

            for row in det:

                x1, y1, x2, y2 = map(float, row[:4])
                score = float(row[4] * row[5])

                # Clip predicted box to the image
                x1, x2 = np.clip([x1, x2], 0, 640)
                y1, y2 = np.clip([y1, y2], 0, 640)

                if x2 <= x1 or y2 <= y1:
                    continue

                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2

                left = int(np.clip(round(cx - 256), 0, 128))
                top = int(np.clip(round(cy - 256), 0, 128))

                crop = gray[top:top+512, left:left+512]

                prompt = np.zeros((512, 512), dtype=np.float32)

                bx1 = int(np.clip(np.floor(x1-left)-1, 0, 512))
                by1 = int(np.clip(np.floor(y1-top)-1, 0, 512))
                bx2 = int(np.clip(np.ceil(x2-left)+1, 0, 512))
                by2 = int(np.clip(np.ceil(y2-top)+1, 0, 512))

                prompt[by1:by2, bx1:bx2] = 1.0

                prompts_and_crops.append(
                    np.stack([crop, prompt], axis=0)
                )

                crop_origins.append((left, top))
                candidate_scores.append(score)

            # Process instances in small batches to limit GPU memory
            for start in range(0, len(prompts_and_crops), 4):

                end = start + 4

                batch = torch.from_numpy(
                    np.stack(prompts_and_crops[start:end])
                ).float().to(device)

                probability = torch.sigmoid(segmenter(batch))

                local_masks = (
                    (probability > 0.5)
                    & (batch[:, 1:2] > 0.5)
                ).cpu().numpy()

                for j, local_mask in enumerate(local_masks):

                    item_index = start + j
                    left, top = crop_origins[item_index]

                    full = np.zeros((640, 640), dtype=bool)

                    full[top:top+512, left:left+512] = (
                        local_mask[0].astype(bool)
                    )

                    if full.any():
                        masks.append(full)
                        scores.append(candidate_scores[item_index])

        evaluation_predictions[filename] = {
            "masks": masks,
            "scores": scores
        }

        if index == 1 or index % 5 == 0:
            print(
                f"Processed {index:02d}/20 | "
                f"{filename} | "
                f"Nonempty masks: {len(masks)}"
            )

print("\nTiles processed:", len(evaluation_predictions))
print(
    "Total nonempty predicted masks:",
    sum(
        len(item["masks"])
        for item in evaluation_predictions.values()
    )
)

assert len(evaluation_predictions) == 20

print("PASS: Automatic predictions cached for 20 validation tiles.")

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

evaluation_ground_truth = {}

for info in evaluation_tiles:
    filename = info["file_name"]

    # Recover the tile index used in Step 57
    tile_index = int(
        filename.replace("val_tile_", "").replace(".jpg", "")
    ) - 1

    source_name, left, top, original_anns = validation_tiles[tile_index]

    masks = []

    for ann in original_anns:
        mask = Image.new("L", (640, 640), 0)
        draw = ImageDraw.Draw(mask)

        for polygon in ann["segmentation"]:
            points = list(zip(
                [v - left for v in polygon[0::2]],
                [v - top for v in polygon[1::2]]
            ))

            if len(points) >= 3:
                draw.polygon(points, fill=1)

        masks.append(np.asarray(mask, dtype=bool))

    evaluation_ground_truth[filename] = masks

# Verify matching filenames and instance counts
assert set(evaluation_ground_truth) == set(evaluation_predictions)

total_gt = sum(
    len(masks) for masks in evaluation_ground_truth.values()
)

empty_gt = sum(
    not mask.any()
    for masks in evaluation_ground_truth.values()
    for mask in masks
)

print("Ground-truth tiles:", len(evaluation_ground_truth))
print("Ground-truth instance masks:", total_gt)
print("Empty ground-truth masks:", empty_gt)
print(
    "Automatic prediction masks:",
    sum(len(v["masks"]) for v in evaluation_predictions.values())
)

assert total_gt == 27
assert empty_gt == 0

print("PASS: Expert polygons aligned with the evaluation tiles.")

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def mask_iou(a, b):
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(intersection / union) if union else 0.0


def mask_nms(masks, scores, threshold):
    if not masks:
        return []

    order = np.argsort(-np.asarray(scores))
    keep = []

    while len(order):
        current = int(order[0])
        keep.append(current)

        survivors = []

        for j in order[1:]:
            j = int(j)

            if mask_iou(masks[current], masks[j]) <= threshold:
                survivors.append(j)

        order = np.asarray(survivors, dtype=int)

    return keep


def evaluate_threshold(threshold):
    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_matched_iou = 0.0
    total_retained = 0

    for info in evaluation_tiles:
        filename = info["file_name"]

        pred_data = evaluation_predictions[filename]
        masks = pred_data["masks"]
        scores = pred_data["scores"]
        gt_masks = evaluation_ground_truth[filename]

        keep = mask_nms(masks, scores, threshold)
        pred_masks = [masks[i] for i in keep]

        total_retained += len(pred_masks)

        matrix = np.zeros(
            (len(pred_masks), len(gt_masks)),
            dtype=np.float32
        )

        for i, pred in enumerate(pred_masks):
            for j, gt in enumerate(gt_masks):
                matrix[i, j] = mask_iou(pred, gt)

        # One-to-one matching among pairs meeting IoU >= 0.5.
        # Hungarian assignment on a binary validity matrix
        # prioritizes the number of valid matches.
        valid = matrix >= 0.5

        rows, cols = linear_sum_assignment(
            -(valid.astype(np.float64) * 2.0 + matrix)
        )

        matched = [
            float(matrix[r, c])
            for r, c in zip(rows, cols)
            if matrix[r, c] >= 0.5
        ]

        tp = len(matched)
        fp = len(pred_masks) - tp
        fn = len(gt_masks) - tp

        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_matched_iou += sum(matched)

    denominator = (
        total_tp + 0.5 * total_fp + 0.5 * total_fn
    )

    pq = total_matched_iou / denominator if denominator else 0.0
    sq = total_matched_iou / total_tp if total_tp else 0.0

    return (
        total_retained, total_tp, total_fp,
        total_fn, sq, pq
    )


print("MULTI-TILE MASK NMS EVALUATION")
print("-" * 73)
print("Threshold | Masks | TP | FP | FN |   SQ   |   PQ")

results_by_threshold = {}

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    result = evaluate_threshold(threshold)
    results_by_threshold[threshold] = result

    n, tp, fp, fn, sq, pq = result

    print(
        f"   {threshold:.1f}    |"
        f"  {n:3d}  |"
        f" {tp:2d} |"
        f" {fp:2d} |"
        f" {fn:2d} |"
        f" {sq:.4f} |"
        f" {pq:.4f}"
    )

print("\nPASS: Multi-tile instance evaluation completed.")

In [ ]:
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from yolox.utils import postprocess

detector = model100.to("cuda").eval()

gt_detection_ious = []
gt_segmentation_ious = []

with torch.inference_mode():
    for info in evaluation_tiles:
        filename = info["file_name"]

        rgb = np.asarray(
            Image.open(root / "val2017" / filename).convert("RGB"),
            dtype=np.float32
        )

        tensor = torch.from_numpy(
            np.ascontiguousarray(rgb.transpose(2, 0, 1))
        ).unsqueeze(0).to("cuda")

        detections = postprocess(
            detector(tensor),
            num_classes=1,
            conf_thre=0.05,
            nms_thre=0.5,
            class_agnostic=True
        )[0]

        boxes = (
            detections[:, :4].cpu().numpy()
            if detections is not None
            else np.empty((0, 4))
        )

        # Recover original tile-local ground-truth boxes
        tile_index = int(
            filename.replace("val_tile_", "").replace(".jpg", "")
        ) - 1

        _, left, top, anns = validation_tiles[tile_index]

        predicted_masks = evaluation_predictions[filename]["masks"]
        ground_truth_masks = evaluation_ground_truth[filename]

        for ann, gt_mask in zip(anns, ground_truth_masks):
            x, y, w, h = ann["bbox"]

            gt_box = np.array([
                x - left,
                y - top,
                x + w - left,
                y + h - top
            ])

            best_box_iou = max(
                (box_iou_xyxy(b, gt_box) for b in boxes),
                default=0.0
            )

            best_mask_iou = max(
                (mask_iou(m, gt_mask) for m in predicted_masks),
                default=0.0
            )

            gt_detection_ious.append(best_box_iou)
            gt_segmentation_ious.append(best_mask_iou)

box_ious = np.asarray(gt_detection_ious)
mask_ious = np.asarray(gt_segmentation_ious)

print("DETECTION VS SEGMENTATION DIAGNOSTIC")
print("-" * 55)

print("Ground-truth instances:", len(box_ious))
print("GT with detection IoU >= 0.5:", int((box_ious >= 0.5).sum()))
print("GT with mask IoU >= 0.5:", int((mask_ious >= 0.5).sum()))

print("Mean best box IoU:", round(float(box_ious.mean()), 4))
print("Mean best mask IoU:", round(float(mask_ious.mean()), 4))

print(
    "Good box, poor mask:",
    int(((box_ious >= 0.5) & (mask_ious < 0.5)).sum())
)

print(
    "Poor box, poor mask:",
    int(((box_ious < 0.5) & (mask_ious < 0.5)).sum())
)

assert len(box_ious) == 27
print("PASS: Stage-wise failure analysis completed.")

In [ ]:
import numpy as np

# Recover GT box areas in exactly the same order as Step 86
gt_box_areas = []

for info in evaluation_tiles:
    filename = info["file_name"]

    tile_index = int(
        filename.replace("val_tile_", "").replace(".jpg", "")
    ) - 1

    _, _, _, anns = validation_tiles[tile_index]

    for ann in anns:
        x, y, w, h = ann["bbox"]
        gt_box_areas.append(float(w * h))

areas = np.asarray(gt_box_areas)

assert len(areas) == len(box_ious) == len(mask_ious) == 27

# Split into three equal-count groups by box area
ordered = np.argsort(areas)
groups = np.array_split(ordered, 3)

print("DETECTION PERFORMANCE BY FILAMENT SIZE")
print("-" * 65)

for label, indices in zip(
    ["Small", "Medium", "Large"], groups
):
    print(f"\n{label} filaments")
    print("Instances:", len(indices))
    print("Median box area:",
          round(float(np.median(areas[indices])), 1), "px²")
    print("Mean detection IoU:",
          round(float(box_ious[indices].mean()), 4))
    print("Detected at IoU >= 0.5:",
          int((box_ious[indices] >= 0.5).sum()),
          "/", len(indices))
    print("Masks at IoU >= 0.5:",
          int((mask_ious[indices] >= 0.5).sum()),
          "/", len(indices))

print("\nPASS: Size-based detection analysis completed.")

In [ ]:
from collections import Counter
import numpy as np

# clean_tiles was prepared in Step 53
# Each entry: (filename, left, top, complete_annotations)

tiles_per_photo = Counter(
    filename
    for filename, left, top, anns in clean_tiles
)

boxes_per_photo = Counter()

for filename, left, top, anns in clean_tiles:
    boxes_per_photo[filename] += len(anns)

tile_counts = np.array(list(tiles_per_photo.values()))
box_counts = np.array(list(boxes_per_photo.values()))

print("FULL DETECTOR TRAINING DIVERSITY")
print("-" * 50)

print("Clean positive tiles:", len(clean_tiles))
print("Unique source photographs:", len(tiles_per_photo))

print("Median positive tiles per photograph:",
      float(np.median(tile_counts)))

print("Maximum positive tiles from one photograph:",
      int(tile_counts.max()))

print("Total box occurrences:",
      int(box_counts.sum()))

print("Photographs contributing at least 10 tiles:",
      int((tile_counts >= 10).sum()))

print("Top 5 contributing photographs:")
for filename, count in tiles_per_photo.most_common(5):
    print(filename, "| Tiles:", count)

assert len(clean_tiles) == 3062

print("\nPASS: Full detector training diversity checked.")

In [ ]:
import numpy as np
from collections import defaultdict, Counter
from pathlib import Path

rng = np.random.default_rng(2026)

# Reuse clean positive tiles from Step 53
full_positive_tiles = list(clean_tiles)

# Reuse clean negatives from Step 54
negatives_by_photo = defaultdict(list)

for tile in negative_tiles:
    filename, left, top, anns = tile
    negatives_by_photo[filename].append(tile)

# Sample negative tiles in rounds across photographs,
# reducing overrepresentation from individual images.
for filename in negatives_by_photo:
    rng.shuffle(negatives_by_photo[filename])

photo_names = sorted(negatives_by_photo)
full_negative_tiles = []

while len(full_negative_tiles) < 1000:
    rng.shuffle(photo_names)
    added = 0

    for filename in photo_names:
        if negatives_by_photo[filename]:
            full_negative_tiles.append(
                negatives_by_photo[filename].pop()
            )
            added += 1

            if len(full_negative_tiles) == 1000:
                break

    if added == 0:
        raise RuntimeError("Insufficient negative tiles")

full_detector_tiles = (
    full_positive_tiles + full_negative_tiles
)

rng.shuffle(full_detector_tiles)

source_counts = Counter(
    tile[0] for tile in full_detector_tiles
)

print("FULL YOLOX TRAINING MANIFEST")
print("-" * 50)

print("Positive tiles:", len(full_positive_tiles))
print("Negative tiles:", len(full_negative_tiles))
print("Total training tiles:", len(full_detector_tiles))

print("Unique source photographs:", len(source_counts))
print("Maximum tiles from one photograph:",
      max(source_counts.values()))

print("Total annotated box occurrences:",
      sum(len(tile[3]) for tile in full_detector_tiles))

assert len(full_positive_tiles) == 3062
assert len(full_negative_tiles) == 1000
assert len(full_detector_tiles) == 4062

print("\nPASS: Full detector training manifest prepared.")

In [ ]:
import json
from pathlib import Path
from PIL import Image
from collections import Counter

full_root = Path("/kaggle/working/v6a_detector_full")
full_image_dir = full_root / "train2017"
full_ann_dir = full_root / "annotations"

full_image_dir.mkdir(parents=True, exist_ok=True)
full_ann_dir.mkdir(parents=True, exist_ok=True)

# Safety: do not overwrite an existing full export
full_json_path = full_ann_dir / "train.json"
assert not full_json_path.exists(), (
    "Full training export already exists. "
    "Inspect it before running this cell again."
)

full_coco = {
    "images": [],
    "annotations": [],
    "categories": [
        {
            "id": 1,
            "name": "filament",
            "supercategory": "solar"
        }
    ]
}

annotation_id = 1

for image_id, (filename, left, top, anns) in enumerate(
    full_detector_tiles, start=1
):
    output_name = f"full_tile_{image_id:05d}.jpg"

    with Image.open(source_files[filename]) as source:
        tile = source.convert("RGB").crop(
            (left, top, left + 640, top + 640)
        )
        tile.save(
            full_image_dir / output_name,
            quality=95
        )

    full_coco["images"].append({
        "id": image_id,
        "file_name": output_name,
        "width": 640,
        "height": 640
    })

    for ann in anns:
        x, y, w, h = map(float, ann["bbox"])

        local_x = x - left
        local_y = y - top

        assert local_x >= -1e-5
        assert local_y >= -1e-5
        assert local_x + w <= 640 + 1e-5
        assert local_y + h <= 640 + 1e-5

        full_coco["annotations"].append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": 1,
            "bbox": [local_x, local_y, w, h],
            "area": w * h,
            "iscrowd": 0
        })

        annotation_id += 1

    if image_id % 1000 == 0:
        print(f"Exported {image_id}/4062 tiles")

# Write JSON only after the image export is complete
with open(full_json_path, "w") as f:
    json.dump(full_coco, f)

# Read back and verify the export
with open(full_json_path) as f:
    exported_full = json.load(f)

counts = Counter(
    ann["image_id"]
    for ann in exported_full["annotations"]
)

positive = sum(
    counts[im["id"]] > 0
    for im in exported_full["images"]
)

negative = sum(
    counts[im["id"]] == 0
    for im in exported_full["images"]
)

saved_images = len(list(full_image_dir.glob("*.jpg")))

print("\nFULL DATASET EXPORT")
print("-" * 45)
print("Training images:", len(exported_full["images"]))
print("Positive tiles:", positive)
print("Negative tiles:", negative)
print("Bounding-box occurrences:",
      len(exported_full["annotations"]))
print("Image files on disk:", saved_images)
print("COCO JSON:", full_json_path)

assert len(exported_full["images"]) == 4062
assert len(exported_full["annotations"]) == 4711
assert positive == 3062
assert negative == 1000
assert saved_images == 4062

print("PASS: Full YOLOX training dataset exported.")

In [ ]:
import json
import shutil
from pathlib import Path
from pycocotools.coco import COCO

full_root = Path("/kaggle/working/v6a_detector_full")
annotation_path = full_root / "annotations/train.json"
image_dir = full_root / "train2017"

# Load through the official COCO interface
full_coco_api = COCO(str(annotation_path))

image_ids = full_coco_api.getImgIds()
annotation_ids = full_coco_api.getAnnIds()

# Verify every annotation points to an existing image
image_id_set = set(image_ids)

assert all(
    ann["image_id"] in image_id_set
    for ann in full_coco_api.loadAnns(annotation_ids)
)

# Verify actual image files
missing = [
    im["file_name"]
    for im in full_coco_api.loadImgs(image_ids)
    if not (image_dir / im["file_name"]).is_file()
]

assert not missing, f"Missing images: {missing[:5]}"

# Preserve the trained pilot checkpoint alongside the new dataset
source_checkpoint = Path(
    "/kaggle/working/v6a_yolox_tiny_pilot100.pth"
)
backup_checkpoint = full_root / "yolox_tiny_pilot100_backup.pth"

assert source_checkpoint.is_file(), (
    "Pilot checkpoint not found; check its path."
)

shutil.copy2(source_checkpoint, backup_checkpoint)

print("\nFULL TRAINING READINESS CHECK")
print("-" * 48)
print("COCO images:", len(image_ids))
print("COCO annotations:", len(annotation_ids))
print("Missing images:", len(missing))
print("Pilot checkpoint backup:", backup_checkpoint)
print(
    "Checkpoint size:",
    round(backup_checkpoint.stat().st_size / 1024**2, 2),
    "MB"
)

assert len(image_ids) == 4062
assert len(annotation_ids) == 4711

print("\nPASS: Dataset readable and pilot checkpoint backed up.")

In [ ]:
import tarfile
import hashlib
from pathlib import Path

working = Path("/kaggle/working")

archive_path = working / "v6a_yolox_full_training_assets.tar.gz"

assets = [
    working / "v6a_detector_full",
    working / "v6a_detector_pilot" / "val2017",
    working / "v6a_detector_pilot" / "annotations" / "val.json",
    working / "v6a_yolox_tiny_pilot20.pth",
    working / "v6a_yolox_tiny_pilot100.pth",
]

for path in assets:
    assert path.exists(), f"Missing asset: {path}"

assert not archive_path.exists(), (
    "Archive already exists. Inspect it before recreating."
)

print("Creating persistent training archive...")

with tarfile.open(archive_path, "w:gz") as tar:
    for path in assets:
        print("Adding:", path.relative_to(working))
        tar.add(path, arcname=str(path.relative_to(working)))

# Verify that archive contents are readable
with tarfile.open(archive_path, "r:gz") as tar:
    members = tar.getmembers()
    archived_files = sum(m.isfile() for m in members)

checksum = hashlib.sha256()

with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
        checksum.update(chunk)

print("\nARCHIVE VERIFICATION")
print("-" * 50)
print("Archive:", archive_path)
print("Size:", round(archive_path.stat().st_size / 1024**3, 3), "GB")
print("Files inside:", archived_files)
print("SHA256:", checksum.hexdigest())

assert archived_files >= 4062

print("\nPASS: Training assets packaged successfully.")

In [ ]:
from pathlib import Path
import os
import subprocess

archive = Path(
    "/kaggle/working/v6a_yolox_full_training_assets.tar.gz"
)

assert archive.exists(), "Training archive not found."

print("Archive size:", round(archive.stat().st_size / 1024**3, 3), "GB")

# Check Kaggle API authentication without printing secrets
try:
    import kaggle
    print("Kaggle API authentication: available")
except Exception as e:
    print("Kaggle API authentication: not configured")
    print("Details:", type(e).__name__, str(e)[:300])

# Check the current Kaggle username
username = os.environ.get("KAGGLE_USERNAME", "")

print("Kaggle username:", username if username else "Not available")

print("\nPASS: Archive verified. Authentication status checked.")

In [ ]:
import subprocess

result = subprocess.run(
    ["kaggle", "datasets", "list", "--mine", "-p", "1"],
    capture_output=True,
    text=True,
    timeout=60
)

print("KAGGLE API AUTHENTICATION TEST")
print("-" * 45)
print("Exit code:", result.returncode)

if result.returncode == 0:
    print("PASS: Kaggle API authenticated successfully.")
    print("We can proceed with private dataset creation.")
else:
    print("Authentication or API request failed.")
    print("Error type/details:")
    print(result.stderr[:500] or result.stdout[:500])

In [ ]:
import subprocess
from pathlib import Path

upload_dir = Path("/kaggle/working/v6a_private_upload")
archive = upload_dir / "v6a_yolox_full_training_assets.tar.gz"
metadata = upload_dir / "dataset-metadata.json"

assert archive.is_file(), "Archive missing"
assert metadata.is_file(), "Dataset metadata missing"

print("Uploading private dataset to Kaggle...")

result = subprocess.run(
    [
        "kaggle", "datasets", "create",
        "-p", str(upload_dir),
        "--dir-mode", "skip"
    ],
    capture_output=True,
    text=True,
    timeout=1200
)

print("\nKAGGLE DATASET CREATION")
print("-" * 50)
print("Exit code:", result.returncode)
print(result.stdout[-2000:])
print(result.stderr[-1000:])

if result.returncode == 0:
    print("\nPASS: Private dataset creation request succeeded.")
    print(
        "Dataset URL:",
        "https://www.kaggle.com/datasets/"
        "asaadwahag/solar-filament-v6a-yolox-assets"
    )
else:
    print("\nUpload failed. Check the error above.")

In [ ]:
import subprocess

dataset_id = "asaadwahag/solar-filament-v6a-yolox-assets"

result = subprocess.run(
    ["kaggle", "datasets", "files", dataset_id],
    capture_output=True,
    text=True,
    timeout=90
)

print("KAGGLE DATASET VERIFICATION")
print("-" * 50)
print("Exit code:", result.returncode)
print(result.stdout[-1500:])
print(result.stderr[-500:])

if result.returncode == 0:
    print("\nPASS: Dataset exists on Kaggle.")
else:
    print("\nDataset not confirmed.")
    print("Your downloaded archive is still a valid backup.")

In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")

print("KAGGLE INPUT DIAGNOSTIC")
print("-" * 45)

folders = sorted(input_root.iterdir())

print("Mounted datasets:")
for folder in folders:
    print(" -", folder.name)

matches = [
    p for p in input_root.rglob("train.json")
    if "v6a" in str(p).lower()
    or "solar-filament-v6a" in str(p).lower()
]

archives = list(
    input_root.rglob("v6a_yolox_full_training_assets.tar.gz")
)

print("\nMatching training annotations:", len(matches))
print("Matching archives:", len(archives))

for path in matches + archives:
    print("FOUND:", path)

if matches or archives:
    print("\nPASS: Persistent assets found.")
else:
    print("\nDataset is not mounted in this notebook.")
    print("Open Add Input → Your Datasets.")
    print("Select solar-filament-v6a-yolox-assets.")
    print("Click Add, then rerun this cell.")

In [ ]:
from pathlib import Path
import json

persistent_root = Path(
    "/kaggle/input/datasets/asaadwahag/"
    "solar-filament-v6a-yolox-assets"
)

training_root = persistent_root / "v6a_detector_full"
checkpoint = persistent_root / "v6a_yolox_tiny_pilot100.pth"
backup_checkpoint = training_root / "yolox_tiny_pilot100_backup.pth"

annotation_file = training_root / "annotations/train.json"
image_dir = training_root / "train2017"

assert annotation_file.is_file(), "Training annotations missing"
assert image_dir.is_dir(), "Training images missing"

with open(annotation_file) as f:
    training_data = json.load(f)

image_count = len(training_data["images"])
annotation_count = len(training_data["annotations"])

available_checkpoint = (
    checkpoint if checkpoint.is_file()
    else backup_checkpoint if backup_checkpoint.is_file()
    else None
)

print("PERSISTENT YOLOX ASSET VERIFICATION")
print("-" * 50)
print("Training images:", image_count)
print("Bounding-box annotations:", annotation_count)
print("Checkpoint available:", available_checkpoint is not None)

if available_checkpoint:
    print("Checkpoint:", available_checkpoint)
    print(
        "Checkpoint size:",
        round(available_checkpoint.stat().st_size / 1024**2, 2),
        "MB"
    )

assert image_count == 4062
assert annotation_count == 4711
assert available_checkpoint is not None

print("\nPASS: Persistent assets ready for expanded training.")

In [ ]:
import copy
import torch
from pathlib import Path

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

persistent_root = Path(
    "/kaggle/input/datasets/asaadwahag/"
    "solar-filament-v6a-yolox-assets"
)

checkpoint_path = (
    persistent_root / "v6a_yolox_tiny_pilot100.pth"
)

assert checkpoint_path.is_file()
assert device.type == "cuda", "Enable Kaggle GPU acceleration."

# Create an independent model, rather than reusing exp's cached model.
detector_full = copy.deepcopy(model100).to(device)

checkpoint_data = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Support either a raw state_dict or a wrapped checkpoint.
state_dict = (
    checkpoint_data["model"]
    if isinstance(checkpoint_data, dict)
    and "model" in checkpoint_data
    else checkpoint_data
)

load_result = detector_full.load_state_dict(
    state_dict,
    strict=True
)

# Verify independence from the original pilot model.
pilot_param = next(model100.parameters())
full_param = next(detector_full.parameters())

assert pilot_param.data_ptr() != full_param.data_ptr()

# Verify numerical equality with checkpoint weights.
checkpoint_first_key = next(iter(state_dict))

assert torch.equal(
    detector_full.state_dict()[checkpoint_first_key].cpu(),
    state_dict[checkpoint_first_key].cpu()
)

print("EXPANDED YOLOX TRAINING INITIALISATION")
print("-" * 55)
print("Device:", device)
print("Checkpoint:", checkpoint_path.name)
print(
    "Trainable parameters:",
    sum(p.numel() for p in detector_full.parameters())
)
print("Missing keys:", len(load_result.missing_keys))
print("Unexpected keys:", len(load_result.unexpected_keys))
print("Independent from pilot model: True")
print("Checkpoint weights verified: True")

print("\nPASS: Expanded detector initialised successfully.")

In [ ]:
import json
import numpy as np
import torch

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

class FullYOLOXTileDataset(Dataset):
    def __init__(self, dataset_root, max_labels=100):
        self.root = Path(dataset_root)
        self.image_dir = self.root / "train2017"
        self.max_labels = max_labels

        with open(self.root / "annotations/train.json") as f:
            data = json.load(f)

        self.images = data["images"]

        self.annotations = defaultdict(list)
        for ann in data["annotations"]:
            self.annotations[ann["image_id"]].append(ann)

        self.max_objects = max(
            len(self.annotations[im["id"]])
            for im in self.images
        )

        assert self.max_objects <= self.max_labels, (
            "Increase max_labels to avoid dropping annotations."
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        info = self.images[index]

        rgb = np.asarray(
            Image.open(
                self.image_dir / info["file_name"]
            ).convert("RGB"),
            dtype=np.float32
        )

        # YOLOX expects RGB, CHW, 0–255
        image = torch.from_numpy(
            np.ascontiguousarray(rgb.transpose(2, 0, 1))
        )

        # YOLOX training target:
        # [class_id, center_x, center_y, width, height]
        # All coordinates in pixels.
        labels = np.zeros(
            (self.max_labels, 5),
            dtype=np.float32
        )

        anns = self.annotations[info["id"]]

        for i, ann in enumerate(anns):
            x, y, w, h = map(float, ann["bbox"])

            labels[i] = [
                0.0,
                x + w / 2,
                y + h / 2,
                w,
                h
            ]

        return image, torch.from_numpy(labels)


full_train_dataset = FullYOLOXTileDataset(
    persistent_root / "v6a_detector_full"
)

full_train_loader = DataLoader(
    full_train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

images, labels = next(iter(full_train_loader))

print("FULL YOLOX DATALOADER CHECK")
print("-" * 50)
print("Training tiles:", len(full_train_dataset))
print("Batches per epoch:", len(full_train_loader))
print("Image batch shape:", tuple(images.shape))
print("Label batch shape:", tuple(labels.shape))
print("Image value range:",
      float(images.min()), "to", float(images.max()))
print("Maximum boxes in one tile:",
      full_train_dataset.max_objects)
print("Objects in test batch:",
      (labels[:, :, 3] > 0).sum(dim=1).tolist())

assert len(full_train_dataset) == 4062
assert images.shape == (2, 3, 640, 640)
assert labels.shape == (2, 100, 5)
assert images.max() <= 255
assert images.min() >= 0

print("\nPASS: Expanded training DataLoader ready.")

In [ ]:
import time
import numpy as np
import torch
from pathlib import Path

TRAINING_STEPS = 200
LEARNING_RATE = 5e-5

detector_full.train()

# YOLOX may cache image grids between calls.
# Ensure training remains on the selected GPU.
detector_full.to(device)

optimizer = torch.optim.AdamW(
    detector_full.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

training_losses = []
training_iterator = iter(full_train_loader)

start_time = time.time()

print("EXPANDED YOLOX TRAINING")
print("-" * 55)
print("Initial checkpoint: pilot 100 steps")
print("Additional training steps:", TRAINING_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Training tiles:", len(full_train_dataset))
print()

for step in range(1, TRAINING_STEPS + 1):

    try:
        images, labels = next(training_iterator)
    except StopIteration:
        training_iterator = iter(full_train_loader)
        images, labels = next(training_iterator)

    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)

    outputs = detector_full(images, labels)
    loss = outputs["total_loss"]

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite loss at step {step}: {loss.item()}"
        )

    loss.backward()
    optimizer.step()

    training_losses.append(float(loss.item()))

    if step % 25 == 0:
        recent = np.mean(training_losses[-25:])

        print(
            f"Step {step:03d}/{TRAINING_STEPS} | "
            f"Mean loss: {recent:.4f} | "
            f"Elapsed: {(time.time()-start_time)/60:.1f} min"
        )

# Save an independent checkpoint
checkpoint_out = Path(
    "/kaggle/working/v6a_yolox_tiny_full300.pth"
)

torch.save(detector_full.state_dict(), checkpoint_out)

print("\nTRAINING SUMMARY")
print("-" * 55)

print(
    "Mean loss, first 25 steps:",
    round(float(np.mean(training_losses[:25])), 4)
)

print(
    "Mean loss, last 25 steps:",
    round(float(np.mean(training_losses[-25:])), 4)
)

print(
    "Total training time:",
    round((time.time() - start_time) / 60, 2),
    "minutes"
)

print("Checkpoint:", checkpoint_out)
print(
    "Checkpoint size:",
    round(checkpoint_out.stat().st_size / 1024**2, 2),
    "MB"
)

assert len(training_losses) == TRAINING_STEPS
assert checkpoint_out.is_file()

print("\nPASS: Expanded YOLOX training completed.")

In [ ]:
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from yolox.utils import postprocess

# Keep both detectors independent
pilot_detector = model100.to(device).eval()
expanded_detector = detector_full.to(device).eval()

val_dir = Path(
    "/kaggle/working/v6a_detector_pilot/val2017"
)

if not val_dir.exists():
    val_dir = (
        persistent_root / "v6a_detector_pilot/val2017"
    )

assert val_dir.exists(), "Validation tile directory missing."

def box_iou(a, b):
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))

    intersection = max(0, x2-x1) * max(0, y2-y1)

    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])

    union = area_a + area_b - intersection
    return float(intersection / union) if union > 0 else 0.0


def evaluate_detector(model):
    best_ious = []
    detection_counts = []

    with torch.inference_mode():
        for info in evaluation_tiles:
            filename = info["file_name"]

            rgb = np.asarray(
                Image.open(val_dir / filename).convert("RGB"),
                dtype=np.float32
            )

            tensor = torch.from_numpy(
                np.ascontiguousarray(
                    rgb.transpose(2, 0, 1)
                )
            ).unsqueeze(0).to(device)

            detections = postprocess(
                model(tensor),
                num_classes=1,
                conf_thre=0.05,
                nms_thre=0.5,
                class_agnostic=True
            )[0]

            boxes = (
                detections[:, :4].cpu().numpy()
                if detections is not None
                else np.empty((0, 4), dtype=np.float32)
            )

            detection_counts.append(len(boxes))

            tile_index = (
                int(filename.split("_")[-1].split(".")[0]) - 1
            )

            _, left, top, anns = validation_tiles[tile_index]

            for ann in anns:
                x, y, w, h = map(float, ann["bbox"])

                gt_box = [
                    x-left, y-top,
                    x+w-left, y+h-top
                ]

                best = max(
                    (box_iou(b, gt_box) for b in boxes),
                    default=0.0
                )
                best_ious.append(best)

    return np.asarray(best_ious), np.asarray(detection_counts)


pilot_ious, pilot_counts = evaluate_detector(pilot_detector)
full_ious, full_counts = evaluate_detector(expanded_detector)

print("YOLOX DETECTOR COMPARISON")
print("-" * 58)
print(f"{'Metric':<29} {'100 steps':>12} {'300 steps':>12}")

print(
    f"{'GT boxes with IoU >= 0.5':<29}"
    f" {(pilot_ious >= 0.5).sum():>12}"
    f" {(full_ious >= 0.5).sum():>12}"
)

print(
    f"{'Mean best box IoU':<29}"
    f" {pilot_ious.mean():>12.4f}"
    f" {full_ious.mean():>12.4f}"
)

print(
    f"{'Total predicted boxes':<29}"
    f" {pilot_counts.sum():>12}"
    f" {full_counts.sum():>12}"
)

print(
    f"{'Tiles with zero detections':<29}"
    f" {(pilot_counts == 0).sum():>12}"
    f" {(full_counts == 0).sum():>12}"
)

print("\nGT instances improved:",
      int((full_ious > pilot_ious + 1e-6).sum()))
print("GT instances worsened:",
      int((full_ious < pilot_ious - 1e-6).sum()))

assert len(pilot_ious) == len(full_ious) == 27

print("\nPASS: Held-out detector comparison completed.")

In [ ]:
import numpy as np
import torch
from PIL import Image
from yolox.utils import postprocess

detector_full.eval()
model_v6a_pretrained.eval().to(device)

evaluation_predictions_300 = {}

with torch.inference_mode():

    for index, info in enumerate(evaluation_tiles, start=1):

        filename = info["file_name"]

        rgb = np.asarray(
            Image.open(val_dir / filename).convert("RGB"),
            dtype=np.float32
        )

        gray = rgb[:, :, 0] / 255.0

        detector_input = torch.from_numpy(
            np.ascontiguousarray(rgb.transpose(2, 0, 1))
        ).unsqueeze(0).to(device)

        detections = postprocess(
            detector_full(detector_input),
            num_classes=1,
            conf_thre=0.05,
            nms_thre=0.5,
            class_agnostic=True
        )[0]

        masks = []
        scores = []

        if detections is not None:

            prompts = []
            origins = []
            candidate_scores = []

            for row in detections.cpu().numpy():

                x1, y1, x2, y2 = map(float, row[:4])
                score = float(row[4] * row[5])

                x1, x2 = np.clip([x1, x2], 0, 640)
                y1, y2 = np.clip([y1, y2], 0, 640)

                if x2 <= x1 or y2 <= y1:
                    continue

                left = int(np.clip(
                    round((x1 + x2) / 2 - 256), 0, 128
                ))
                top = int(np.clip(
                    round((y1 + y2) / 2 - 256), 0, 128
                ))

                crop = gray[top:top+512, left:left+512]

                prompt = np.zeros((512, 512), dtype=np.float32)

                bx1 = int(np.clip(np.floor(x1-left)-1, 0, 512))
                by1 = int(np.clip(np.floor(y1-top)-1, 0, 512))
                bx2 = int(np.clip(np.ceil(x2-left)+1, 0, 512))
                by2 = int(np.clip(np.ceil(y2-top)+1, 0, 512))

                prompt[by1:by2, bx1:bx2] = 1.0

                prompts.append(np.stack([crop, prompt], axis=0))
                origins.append((left, top))
                candidate_scores.append(score)

            for start in range(0, len(prompts), 4):

                batch = torch.from_numpy(
                    np.stack(prompts[start:start+4])
                ).float().to(device)

                probabilities = torch.sigmoid(
                    model_v6a_pretrained(batch)
                )

                local_masks = (
                    (probabilities > 0.5)
                    & (batch[:, 1:2] > 0.5)
                ).cpu().numpy()

                for j, local_mask in enumerate(local_masks):

                    k = start + j
                    left, top = origins[k]

                    full_mask = np.zeros(
                        (640, 640), dtype=bool
                    )

                    full_mask[
                        top:top+512,
                        left:left+512
                    ] = local_mask[0]

                    if full_mask.any():
                        masks.append(full_mask)
                        scores.append(candidate_scores[k])

        evaluation_predictions_300[filename] = {
            "masks": masks,
            "scores": scores
        }

        if index % 5 == 0:
            print(
                f"Processed {index}/20 | "
                f"Nonempty masks: {len(masks)}"
            )

print("\nAUTOMATIC SEGMENTATION — 300-STEP DETECTOR")
print("-" * 55)
print("Validation tiles:", len(evaluation_predictions_300))

print(
    "Nonempty predicted masks:",
    sum(
        len(v["masks"])
        for v in evaluation_predictions_300.values()
    )
)

assert len(evaluation_predictions_300) == 20

print("\nPASS: Improved detector-to-mask inference completed.")

In [ ]:
# Preserve the original 100-step predictions
predictions_100 = evaluation_predictions
predictions_300 = evaluation_predictions_300

# Step 85's evaluate_threshold() reads this global variable.
# Reuse it to ensure identical matching and scoring.

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]
comparison = {}

print("END-TO-END INSTANCE SEGMENTATION COMPARISON")
print("-" * 79)
print(
    "NMS   | 100-step: TP FP FN   PQ | "
    "300-step: TP FP FN   PQ | Change"
)

for threshold in thresholds:

    evaluation_predictions = predictions_100
    old_result = evaluate_threshold(threshold)

    evaluation_predictions = predictions_300
    new_result = evaluate_threshold(threshold)

    comparison[threshold] = {
        "100_steps": old_result,
        "300_steps": new_result
    }

    _, tp_old, fp_old, fn_old, _, pq_old = old_result
    _, tp_new, fp_new, fn_new, _, pq_new = new_result

    print(
        f"{threshold:.1f}   | "
        f"{tp_old:2d} {fp_old:2d} {fn_old:2d} {pq_old:.4f} | "
        f"{tp_new:2d} {fp_new:2d} {fn_new:2d} {pq_new:.4f} | "
        f"{pq_new - pq_old:+.4f}"
    )

# Restore original variable for subsequent experiments
evaluation_predictions = predictions_100

print("\nPASS: End-to-end pipeline comparison completed.")

In [ ]:
from torch.utils.data import Dataset, DataLoader

print("AVAILABLE INSTANCE DATASETS AND LOADERS")
print("-" * 55)

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if isinstance(obj, (Dataset, DataLoader)):
        try:
            ds = obj.dataset if isinstance(obj, DataLoader) else obj
            print(
                f"{name:35s} | "
                f"{type(obj).__name__:12s} | "
                f"Dataset: {type(ds).__name__:28s} | "
                f"Samples: {len(ds)}"
            )
        except Exception:
            pass

In [ ]:
import torch
import numpy as np
import time
from pathlib import Path

torch.manual_seed(2026)
device = torch.device("cuda")

segmenter = model_v6a_pretrained.to(device)
segmenter.train()

optimizer = torch.optim.AdamW(
    segmenter.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

steps = 300
losses = []
iterator = iter(pilot_loader)
start = time.time()

for step in range(1, steps + 1):
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(pilot_loader)
        batch = next(iterator)

    if len(batch) == 3:
        images, boxes, masks = batch
        inputs = torch.cat([images, boxes], dim=1)
    elif len(batch) == 2:
        inputs, masks = batch
        boxes = inputs[:, 1:2]
    else:
        raise ValueError(f"Unexpected batch structure: {len(batch)}")

    inputs = inputs.float().to(device)
    masks = masks.float().to(device)
    boxes = boxes.float().to(device)

    optimizer.zero_grad(set_to_none=True)

    logits = segmenter(inputs)
    loss_output = box_instance_loss(logits, masks, boxes)

    # Loss function returns (total_loss, auxiliary_values)
    loss = loss_output[0] if isinstance(
        loss_output, (tuple, list)
    ) else loss_output

    if not torch.isfinite(loss).all():
        raise RuntimeError(f"Non-finite loss at step {step}")

    loss.backward()
    optimizer.step()

    losses.append(float(loss.item()))

    if step % 50 == 0:
        print(
            f"Step {step:03d}/300 | "
            f"Mean loss: {np.mean(losses[-50:]):.4f}"
        )

checkpoint = Path(
    "/kaggle/working/v6a_instance_unet_320steps.pth"
)

torch.save(segmenter.state_dict(), checkpoint)

print("\nTRAINING SUMMARY")
print("First 50-step mean:", round(np.mean(losses[:50]), 4))
print("Last 50-step mean:", round(np.mean(losses[-50:]), 4))
print("Training time:", round((time.time()-start)/60, 2), "minutes")
print("Checkpoint:", checkpoint)
print("PASS: U-Net training completed.")

In [ ]:
import numpy as np
import torch
from PIL import Image
from yolox.utils import postprocess

detector_full.eval()
segmenter = model_v6a_pretrained.to(device).eval()

evaluation_predictions_320 = {}

with torch.inference_mode():
    for info in evaluation_tiles:
        filename = info["file_name"]

        rgb = np.asarray(
            Image.open(val_dir / filename).convert("RGB"),
            dtype=np.float32
        )
        gray = rgb[:, :, 0] / 255.0

        x = torch.from_numpy(
            np.ascontiguousarray(rgb.transpose(2, 0, 1))
        ).unsqueeze(0).to(device)

        detections = postprocess(
            detector_full(x),
            num_classes=1,
            conf_thre=0.05,
            nms_thre=0.5,
            class_agnostic=True
        )[0]

        masks, scores = [], []
        prompts, origins, candidate_scores = [], [], []

        if detections is not None:
            for row in detections.cpu().numpy():
                x1, y1, x2, y2 = map(float, row[:4])
                score = float(row[4] * row[5])

                x1, x2 = np.clip([x1, x2], 0, 640)
                y1, y2 = np.clip([y1, y2], 0, 640)

                if x2 <= x1 or y2 <= y1:
                    continue

                left = int(np.clip(
                    round((x1 + x2) / 2 - 256), 0, 128
                ))
                top = int(np.clip(
                    round((y1 + y2) / 2 - 256), 0, 128
                ))

                crop = gray[top:top+512, left:left+512]
                prompt = np.zeros((512, 512), dtype=np.float32)

                bx1 = int(np.clip(np.floor(x1-left)-1, 0, 512))
                by1 = int(np.clip(np.floor(y1-top)-1, 0, 512))
                bx2 = int(np.clip(np.ceil(x2-left)+1, 0, 512))
                by2 = int(np.clip(np.ceil(y2-top)+1, 0, 512))

                prompt[by1:by2, bx1:bx2] = 1.0

                prompts.append(np.stack([crop, prompt]))
                origins.append((left, top))
                candidate_scores.append(score)

        for start in range(0, len(prompts), 4):
            batch = torch.from_numpy(
                np.stack(prompts[start:start+4])
            ).float().to(device)

            probabilities = torch.sigmoid(segmenter(batch))

            local_masks = (
                (probabilities > 0.5)
                & (batch[:, 1:2] > 0.5)
            ).cpu().numpy()

            for j, local_mask in enumerate(local_masks):
                k = start + j
                left, top = origins[k]

                full = np.zeros((640, 640), dtype=bool)
                full[top:top+512, left:left+512] = local_mask[0]

                if full.any():
                    masks.append(full)
                    scores.append(candidate_scores[k])

        evaluation_predictions_320[filename] = {
            "masks": masks,
            "scores": scores
        }

# Compare against the previously cached 20-step U-Net outputs.
# The detector is fixed at 300 steps in both cases.
previous_predictions = evaluation_predictions
comparison_320 = {}

for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
    evaluation_predictions = evaluation_predictions_300
    old_result = evaluate_threshold(threshold)

    evaluation_predictions = evaluation_predictions_320
    new_result = evaluate_threshold(threshold)

    comparison_320[threshold] = (old_result, new_result)

evaluation_predictions = previous_predictions

print("U-NET IMPROVEMENT — FIXED 300-STEP YOLOX")
print("-" * 72)
print("NMS | 20-step U-Net PQ | 320-step U-Net PQ | TP | FP | FN")

for threshold, (old, new) in comparison_320.items():
    _, _, _, _, _, old_pq = old
    _, tp, fp, fn, _, new_pq = new

    print(
        f"{threshold:.1f} | "
        f"{old_pq:.4f}           | "
        f"{new_pq:.4f}            | "
        f"{tp:2d} | {fp:2d} | {fn:2d}"
    )

print("\nPASS: End-to-end U-Net comparison completed.")

In [ ]:
import torch
import json
from pathlib import Path

save_dir = Path("/kaggle/working/v6a_best_models")
save_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    detector_full.state_dict(),
    save_dir / "yolox_tiny_300steps.pth"
)

torch.save(
    model_v6a_pretrained.state_dict(),
    save_dir / "instance_unet_320steps.pth"
)

metadata = {
    "detector": "YOLOX-Tiny, 300 total training steps",
    "segmenter": "BoxUNetV6A, 320 total training steps",
    "pilot_validation_tiles": 20,
    "pilot_ground_truth_instances": 27,
    "mask_nms_threshold": 0.4,
    "pilot_pq": 0.3808,
    "note": "Selected positive-tile validation only; not official Kaggle PQ"
}

with open(save_dir / "experiment.json", "w") as f:
    json.dump(metadata, f, indent=2)

for path in sorted(save_dir.iterdir()):
    print(path.name, round(path.stat().st_size / 1024**2, 2), "MB")

print("\nPASS: Both improved models saved together.")

In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter

# Recover held-out source filenames from existing validation tiles
validation_names = sorted({
    tile[0] for tile in validation_tiles
})

# Count annotated filaments per photograph,
# using one selected annotation record per source image
candidates = [
    name for name in validation_names
    if name in selected_records
    and name in source_files
]

assert candidates, "No validation source photographs found."

selected_name = max(
    candidates,
    key=lambda name: len(
        selected_records[name].get("annotations", [])
    )
)

image_path = Path(source_files[selected_name])

assert image_path.is_file(), f"Image missing: {image_path}"

with Image.open(image_path) as im:
    image_size = im.size

print("FULL-RESOLUTION VALIDATION IMAGE")
print("-" * 50)
print("Filename:", selected_name)
print("Dimensions:", image_size)
print("Path:", image_path)
print("Available validation photographs:", len(validation_names))

print("\nPASS: Held-out photograph selected.")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from PIL import Image
from yolox.utils import postprocess

# Fixed inference settings
TILE = 640
STRIDE = 384
CONF = 0.05
BOX_NMS = 0.5
MASK_NMS = 0.4

detector_full.eval()
model_v6a_pretrained.eval()

rgb_full = np.asarray(
    Image.open(image_path).convert("RGB"),
    dtype=np.float32
)
H, W = rgb_full.shape[:2]

gray_full = rgb_full[:, :, 0] / 255.0

def positions(length, tile, stride):
    p = list(range(0, length - tile + 1, stride))
    if p[-1] != length - tile:
        p.append(length - tile)
    return p

xs = positions(W, TILE, STRIDE)
ys = positions(H, TILE, STRIDE)

# Stage 1: Detect boxes across overlapping tiles
candidates = []

with torch.inference_mode():
    for top in ys:
        for left in xs:
            crop = rgb_full[top:top+TILE, left:left+TILE]

            x = torch.from_numpy(
                np.ascontiguousarray(crop.transpose(2, 0, 1))
            ).unsqueeze(0).to(device)

            det = postprocess(
                detector_full(x),
                num_classes=1,
                conf_thre=CONF,
                nms_thre=BOX_NMS,
                class_agnostic=True
            )[0]

            if det is None:
                continue

            for row in det.cpu().numpy():
                x1, y1, x2, y2 = map(float, row[:4])
                score = float(row[4] * row[5])

                box = np.array([
                    np.clip(x1 + left, 0, W),
                    np.clip(y1 + top, 0, H),
                    np.clip(x2 + left, 0, W),
                    np.clip(y2 + top, 0, H)
                ], dtype=np.float32)

                if box[2] > box[0] and box[3] > box[1]:
                    candidates.append((box, score))

# Global box NMS across overlapping tiles
def box_iou(a, b):
    xx1, yy1 = np.maximum(a[:2], b[:2])
    xx2, yy2 = np.minimum(a[2:], b[2:])
    intersection = max(0, xx2-xx1) * max(0, yy2-yy1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - intersection
    return intersection / union if union else 0.0

order = sorted(
    range(len(candidates)),
    key=lambda i: candidates[i][1],
    reverse=True
)

kept_boxes = []

for idx in order:
    box, score = candidates[idx]
    if all(box_iou(box, old_box) <= BOX_NMS
           for old_box, _ in kept_boxes):
        kept_boxes.append((box, score))

print("Tiles:", len(xs) * len(ys))
print("Raw detections:", len(candidates))
print("Boxes after global NMS:", len(kept_boxes))

# Stage 2: Generate instance masks
full_masks = []
full_scores = []

with torch.inference_mode():
    for box, score in kept_boxes:
        x1, y1, x2, y2 = box

        left = int(np.clip(
            round((x1+x2)/2 - 256), 0, W-512
        ))
        top = int(np.clip(
            round((y1+y2)/2 - 256), 0, H-512
        ))

        crop = gray_full[top:top+512, left:left+512]
        prompt = np.zeros((512, 512), dtype=np.float32)

        bx1 = int(np.clip(np.floor(x1-left)-1, 0, 512))
        by1 = int(np.clip(np.floor(y1-top)-1, 0, 512))
        bx2 = int(np.clip(np.ceil(x2-left)+1, 0, 512))
        by2 = int(np.clip(np.ceil(y2-top)+1, 0, 512))

        prompt[by1:by2, bx1:bx2] = 1.0

        inp = torch.from_numpy(
            np.stack([crop, prompt])
        ).unsqueeze(0).float().to(device)

        probability = torch.sigmoid(
            model_v6a_pretrained(inp)
        )[0, 0].cpu().numpy()

        local_mask = (probability > 0.5) & (prompt > 0.5)

        if not local_mask.any():
            continue

        # Store local masks to avoid unnecessary full-image copies
        full_masks.append((left, top, local_mask))
        full_scores.append(score)

# Stage 3: Global mask NMS, calculating overlap locally
def local_mask_iou(a, b):
    ax, ay, am = a
    bx, by, bm = b

    x1, y1 = max(ax, bx), max(ay, by)
    x2 = min(ax+512, bx+512)
    y2 = min(ay+512, by+512)

    intersection = 0

    if x2 > x1 and y2 > y1:
        intersection = np.logical_and(
            am[y1-ay:y2-ay, x1-ax:x2-ax],
            bm[y1-by:y2-by, x1-bx:x2-bx]
        ).sum()

    union = am.sum() + bm.sum() - intersection
    return intersection / union if union else 0.0

order = np.argsort(-np.asarray(full_scores))
keep = []

while len(order):
    current = int(order[0])
    keep.append(current)

    order = np.asarray([
        int(j) for j in order[1:]
        if local_mask_iou(
            full_masks[current], full_masks[int(j)]
        ) <= MASK_NMS
    ], dtype=int)

# Visualization
overlay = rgb_full.astype(np.uint8).copy()

rng = np.random.default_rng(2026)
colors = rng.integers(60, 256, size=(len(keep), 3))

for i, k in enumerate(keep):
    left, top, mask = full_masks[k]
    region = overlay[top:top+512, left:left+512]
    region[mask] = (
        0.4 * region[mask] + 0.6 * colors[i]
    ).astype(np.uint8)

plt.figure(figsize=(11, 11))
plt.imshow(overlay)
plt.title(
    f"V6-A automatic full-image segmentation — "
    f"{len(keep)} instances"
)
plt.axis("off")
plt.show()

print("\nFULL-IMAGE INFERENCE")
print("Nonempty masks:", len(full_masks))
print("Instances after Mask NMS:", len(keep))
print("PASS: Full-resolution inference completed.")

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment

# Recover the selected expert annotation record
record = selected_records[selected_name]

# Support either annotations embedded in the record
# or stored separately in the original annotation list.
gt_annotations = record.get("annotations", None)

if gt_annotations is None:
    gt_annotations = [
        ann for ann in data["annotations"]
        if ann["image_id"] == record["id"]
    ]

# Rasterize original expert polygons
gt_masks_full = []

for ann in gt_annotations:
    canvas = Image.new("1", (W, H), 0)
    draw = ImageDraw.Draw(canvas)

    for polygon in ann["segmentation"]:
        points = list(zip(
            polygon[0::2],
            polygon[1::2]
        ))
        if len(points) >= 3:
            draw.polygon(points, fill=1)

    mask = np.asarray(canvas, dtype=bool)

    if mask.any():
        gt_masks_full.append(mask)

# Convert retained local masks into full-image masks
pred_masks_full = []

for k in keep:
    left, top, local_mask = full_masks[k]

    mask = np.zeros((H, W), dtype=bool)
    mask[top:top+512, left:left+512] = local_mask

    pred_masks_full.append(mask)

# Compute pairwise mask IoU
matrix = np.zeros(
    (len(pred_masks_full), len(gt_masks_full)),
    dtype=np.float32
)

for i, pred in enumerate(pred_masks_full):
    for j, gt in enumerate(gt_masks_full):
        matrix[i, j] = mask_iou(pred, gt)

# Match at IoU >= 0.5
valid = matrix >= 0.5

rows, cols = linear_sum_assignment(
    -(valid.astype(float) * 2.0 + matrix)
)

matched = [
    float(matrix[r, c])
    for r, c in zip(rows, cols)
    if matrix[r, c] >= 0.5
]

tp = len(matched)
fp = len(pred_masks_full) - tp
fn = len(gt_masks_full) - tp

denominator = tp + 0.5 * fp + 0.5 * fn
pq = sum(matched) / denominator if denominator else 0.0

print("FULL-RESOLUTION INSTANCE EVALUATION")
print("-" * 55)
print("Image:", selected_name)
print("Ground-truth filaments:", len(gt_masks_full))
print("Predicted filaments:", len(pred_masks_full))
print("TP:", tp)
print("FP:", fp)
print("FN:", fn)
print("PQ:", round(pq, 4))
print("Matched IoUs:", np.round(matched, 3).tolist())
print("\nPASS: Full-image evaluation completed.")

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

# Reuse the full-image IoU matrix from Step 110
valid = matrix >= 0.5

rows, cols = linear_sum_assignment(
    -(valid.astype(float) * 2.0 + matrix)
)

matched_predictions = {
    int(r) for r, c in zip(rows, cols)
    if valid[r, c]
}

print("FULL-IMAGE FALSE-POSITIVE ANALYSIS")
print("-" * 55)

fragment_candidates = 0
unrelated_candidates = 0

for i in range(len(pred_masks_full)):
    if i in matched_predictions:
        continue

    best_gt = int(np.argmax(matrix[i]))
    best_iou = float(matrix[i, best_gt])

    # Also measure how much of the predicted mask
    # lies inside its best-matching expert filament.
    pred = pred_masks_full[i]
    gt = gt_masks_full[best_gt]

    intersection = np.logical_and(pred, gt).sum()
    pred_area = pred.sum()

    overlap_fraction = (
        intersection / pred_area if pred_area else 0.0
    )

    if overlap_fraction >= 0.5:
        fragment_candidates += 1
        category = "Possible filament fragment"
    else:
        unrelated_candidates += 1
        category = "Low expert overlap"

    print(
        f"Prediction {i:02d} | "
        f"Best GT IoU: {best_iou:.3f} | "
        f"Predicted area inside GT: {overlap_fraction:.3f} | "
        f"{category}"
    )

print("\nSUMMARY")
print("Possible filament fragments:", fragment_candidates)
print("Low-overlap predictions:", unrelated_candidates)

assert (
    fragment_candidates + unrelated_candidates == fp
)

print("\nPASS: False-positive analysis completed.")

In [ ]:
import numpy as np
import cv2
from scipy.optimize import linear_sum_assignment

# Existing predictions after global Mask NMS
original_masks = [m.copy() for m in pred_masks_full]

def merge_nearby_masks(masks, gap=5):
    n = len(masks)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[rj] = ri

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (2 * gap + 1, 2 * gap + 1)
    )

    # Merge instances whose masks overlap or lie within
    # the specified spatial gap.
    dilated = [
        cv2.dilate(m.astype(np.uint8), kernel) > 0
        for m in masks
    ]

    for i in range(n):
        for j in range(i + 1, n):
            if np.logical_and(dilated[i], masks[j]).any():
                union(i, j)

    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)

    merged = [
        np.logical_or.reduce([masks[i] for i in indices])
        for indices in groups.values()
    ]

    return merged, list(groups.values())


def full_image_pq(predictions, ground_truth):
    mat = np.zeros(
        (len(predictions), len(ground_truth)),
        dtype=np.float32
    )

    for i, pred in enumerate(predictions):
        for j, gt in enumerate(ground_truth):
            mat[i, j] = mask_iou(pred, gt)

    valid = mat >= 0.5

    rows, cols = linear_sum_assignment(
        -(valid.astype(float) * 2.0 + mat)
    )

    matched = [
        float(mat[r, c])
        for r, c in zip(rows, cols)
        if valid[r, c]
    ]

    tp = len(matched)
    fp = len(predictions) - tp
    fn = len(ground_truth) - tp

    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = sum(matched) / denominator if denominator else 0.0

    return tp, fp, fn, pq


print("FULL-IMAGE MASK MERGING EXPERIMENT")
print("-" * 62)
print("Gap | Instances | TP | FP | FN | PQ")

merge_results = {}

for gap in [0, 3, 5, 10]:
    merged, groups = merge_nearby_masks(
        original_masks, gap=gap
    )

    tp, fp, fn, pq = full_image_pq(
        merged, gt_masks_full
    )

    merge_results[gap] = {
        "masks": merged,
        "groups": groups,
        "pq": pq
    }

    print(
        f"{gap:3d} | {len(merged):9d} | "
        f"{tp:2d} | {fp:2d} | {fn:2d} | {pq:.4f}"
    )

print("\nOriginal PQ:", round(
    full_image_pq(original_masks, gt_masks_full)[-1], 4
))

print("PASS: Full-image mask merging evaluated.")

In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment

# Select 10 held-out photographs reproducibly.
rng = np.random.default_rng(2026)
names = sorted(validation_names)

selected_full_images = rng.choice(
    names, size=10, replace=False
).tolist()

detector_full.eval()
model_v6a_pretrained.eval()

def predict_full_image(path):
    rgb = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32)
    h, w = rgb.shape[:2]
    gray = rgb[:, :, 0] / 255.0

    xs = positions(w, 640, 384)
    ys = positions(h, 640, 384)

    candidates = []

    with torch.inference_mode():
        for top in ys:
            for left in xs:
                tile = rgb[top:top+640, left:left+640]

                x = torch.from_numpy(
                    np.ascontiguousarray(tile.transpose(2, 0, 1))
                ).unsqueeze(0).to(device)

                det = postprocess(
                    detector_full(x),
                    num_classes=1,
                    conf_thre=0.05,
                    nms_thre=0.5,
                    class_agnostic=True
                )[0]

                if det is None:
                    continue

                for row in det.cpu().numpy():
                    box = row[:4].astype(np.float32).copy()
                    box[[0, 2]] += left
                    box[[1, 3]] += top
                    box[[0, 2]] = np.clip(box[[0, 2]], 0, w)
                    box[[1, 3]] = np.clip(box[[1, 3]], 0, h)

                    if box[2] > box[0] and box[3] > box[1]:
                        candidates.append(
                            (box, float(row[4] * row[5]))
                        )

        # Global box NMS
        candidates.sort(key=lambda item: item[1], reverse=True)
        kept = []

        for box, score in candidates:
            if all(box_iou(box, b) <= 0.5 for b, _ in kept):
                kept.append((box, score))

        masks = []

        for box, score in kept:
            x1, y1, x2, y2 = box

            left = int(np.clip(round((x1+x2)/2 - 256), 0, w-512))
            top = int(np.clip(round((y1+y2)/2 - 256), 0, h-512))

            crop = gray[top:top+512, left:left+512]
            prompt = np.zeros((512, 512), dtype=np.float32)

            bx1 = int(np.clip(np.floor(x1-left)-1, 0, 512))
            by1 = int(np.clip(np.floor(y1-top)-1, 0, 512))
            bx2 = int(np.clip(np.ceil(x2-left)+1, 0, 512))
            by2 = int(np.clip(np.ceil(y2-top)+1, 0, 512))

            prompt[by1:by2, bx1:bx2] = 1.0

            inp = torch.from_numpy(
                np.stack([crop, prompt])
            ).unsqueeze(0).float().to(device)

            prob = torch.sigmoid(model_v6a_pretrained(inp))[0, 0]
            local = (prob.cpu().numpy() > 0.5) & (prompt > 0.5)

            if local.any():
                full = np.zeros((h, w), dtype=bool)
                full[top:top+512, left:left+512] = local
                masks.append((full, score))

    # Global mask NMS
    masks.sort(key=lambda item: item[1], reverse=True)
    retained = []

    for mask, score in masks:
        if all(mask_iou(mask, old) <= 0.4 for old in retained):
            retained.append(mask)

    return retained


def ground_truth_full(name):
    record = selected_records[name]
    anns = record.get("annotations")

    if anns is None:
        anns = [
            ann for ann in data["annotations"]
            if ann["image_id"] == record["id"]
        ]

    gt = []

    for ann in anns:
        canvas = Image.new("1", (2048, 2048), 0)
        draw = ImageDraw.Draw(canvas)

        for polygon in ann["segmentation"]:
            points = list(zip(polygon[0::2], polygon[1::2]))
            if len(points) >= 3:
                draw.polygon(points, fill=1)

        mask = np.asarray(canvas, dtype=bool)

        if mask.any():
            gt.append(mask)

    return gt


totals = {
    "original": np.zeros(4, dtype=float),
    "merged": np.zeros(4, dtype=float)
}

# Accumulate TP, FP, FN, and matched IoU sum.
def counts_for_pq(pred, gt):
    mat = np.zeros((len(pred), len(gt)), dtype=np.float32)

    for i, p in enumerate(pred):
        for j, g in enumerate(gt):
            mat[i, j] = mask_iou(p, g)

    valid = mat >= 0.5
    rows, cols = linear_sum_assignment(
        -(valid.astype(float) * 2.0 + mat)
    )

    matched = [
        float(mat[r, c]) for r, c in zip(rows, cols)
        if valid[r, c]
    ]

    tp = len(matched)
    return np.array([
        tp, len(pred)-tp, len(gt)-tp, sum(matched)
    ], dtype=float)


print("FULL-RESOLUTION VALIDATION")
print("-" * 65)

for i, name in enumerate(selected_full_images, 1):
    predictions = predict_full_image(source_files[name])
    gt = ground_truth_full(name)

    merged, _ = merge_nearby_masks(predictions, gap=0)

    totals["original"] += counts_for_pq(predictions, gt)
    totals["merged"] += counts_for_pq(merged, gt)

    print(
        f"{i:02d}/10 | {name} | "
        f"GT={len(gt)} Pred={len(predictions)} Merged={len(merged)}"
    )

print("\nAGGREGATED RESULTS")
print("-" * 65)

for method, (tp, fp, fn, iou_sum) in totals.items():
    denominator = tp + 0.5*fp + 0.5*fn
    pq = iou_sum / denominator if denominator else 0

    print(
        f"{method:8s} | TP={int(tp):3d} "
        f"FP={int(fp):3d} FN={int(fn):3d} "
        f"PQ={pq:.4f}"
    )

print("\nPASS: Full-resolution validation completed.")